# ENSO × chuva: escolha da divisão regional (e, depois, das defasagens)

**Pergunta desta etapa:** qual divisão regional agrupa melhor as microrregiões que **respondem do mesmo jeito** ao ENSO?

O ENSO tem efeito oposto no Sul e no Norte/Nordeste. Uma análise nacional cancela esses sinais, então a divisão regional precisa ser decidida **antes** da varredura de defasagens. A resposta usada aqui é a **chuva mensal** (o mecanismo físico), e não o rendimento:
- **mais amostra:** cerca de 160 meses de safra por microrregião, contra 17 a 26 safras;
- **independe do `desvio_clima`**, que ainda é nacional.

**Divisões avaliadas**, todas definidas *a priori*, sem ajuste aos dados:

| Divisão | Fonte | Observação |
|---|---|---|
| Macrorregião (5) | IBGE / geobr | |
| UF (27) | IBGE / geobr | |
| Bioma | IBGE 2019 / geobr | atribuição pela maior área |
| Região geográfica intermediária | IBGE 2017 / geobr (malha 2020) | atribuição pela maior área |
| Macrorregião + MATOPIBA | composição | MATOPIBA **aproximado** (MA/TO/PI/BA com predomínio de Cerrado) |
| UF (pequenas → macrorregião) | variante | UFs com menos de 3 microrregiões com dados vão para "resto da macrorregião" |
| Bioma (Pantanal → Cerrado) | variante | o Pantanal tem pouquíssimas microrregiões com soja |
| Regiões edafoclimáticas da soja (Embrapa) | arquivo externo | avaliada só se `data/raw/regioes/rec_soja.csv` existir |

**Como as divisões são comparadas:**
- **Seção 5, dentro da amostra (descritiva):** homogeneidade (R² ajustado), coerência de sinal, força e estabilidade do sinal regional e tamanho das regiões. ⚠️ Essas métricas usam os mesmos anos em que as correlações foram calculadas e **favorecem divisões com muitas regiões pequenas**, que se ajustam ao ruído.
- **Seção 6, fora da amostra (critério principal):** a resposta de cada região, estimada com as safras **pares**, prevê a resposta de cada microrregião nas safras **ímpares** (e vice-versa)? É o teste que decide.

In [ ]:
import warnings
from pathlib import Path

import geobr
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

from soja_clima.geo import mapa_estacoes_cd_micro

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)

PROCESSED = Path("../../data/processed")
RAW = Path("../../data/raw")

# Parâmetros da análise (cada um é uma decisão; ver justificativas no texto)
P = dict(
    meses_safra=[9, 10, 11, 12, 1, 2, 3, 4],   # mesma janela set–abr do pipeline
    completude_mensal=0.80,                    # mês da estação só vale com >= 80% dos dias
    min_anos_climatologia=10,                  # anos mínimos para a média/desvio do mês
    so_estacoes_no_poligono=True,              # ignora estações "substitutas" (evita repetir a mesma estação em várias micros)
    so_micros_com_soja=True,                   # avalia só as microrregiões da base de modelagem
    min_micros_regiao=3,                       # regiões menores que isso são "frágeis"
    alpha_fdr=0.05,                            # nível do controle de falsas descobertas (Benjamini–Hochberg)
    lag_robustez=3,                            # defasagem extra só para checar a robustez da escolha
    ano_malha=2020,
    crs_area=5880,                             # SIRGAS 2000 / Brazil Polyconic (áreas em m²)
)

## 1. Dados
- **Chuva:** `clima_diario.parquet` (estações do INMET).
- **ENSO:** `pacifico_indices.parquet` (ONI da NOAA; SOI do BOM, com **sinal oposto**: negativo no El Niño).
- **Peso pela soja:** produção média de cada microrregião na `base_final_modelagem.csv`.
- **Malhas:** geobr.

In [ ]:
clima = pd.read_parquet(PROCESSED / "clima_diario.parquet", columns=["codigo_estacao", "data", "prec_total"])
clima["data"] = pd.to_datetime(clima["data"])

pac = pd.read_parquet(PROCESSED / "pacifico_indices.parquet")
pac["mes_ref"] = pd.to_datetime(dict(year=pac["ano"], month=pac["mes"], day=1)).dt.to_period("M")
enso = (pac.set_index("mes_ref")[["oni", "soi"]].sort_index()
           .reindex(pd.period_range(pac["mes_ref"].min(), pac["mes_ref"].max(), freq="M")))

base = pd.read_csv(PROCESSED / "base_final_modelagem.csv", dtype={"CD_MICRO": str})
producao = base.groupby("CD_MICRO")["producao_ton"].mean()

micro = geobr.read_micro_region(year=P["ano_malha"])
micro["CD_MICRO"] = micro["code_micro"].astype(int).astype(str)
print(f"{clima['codigo_estacao'].nunique()} estações | ENSO {enso.index.min()}–{enso.index.max()} | "
      f"{len(micro)} microrregiões | {len(producao)} com soja na base")

## 2. Chuva mensal por microrregião e anomalia padronizada
1. **Por estação e mês:** soma da chuva, aceita só se ≥ 80% dos dias do mês têm dado (a mesma lógica de completude do pipeline).
2. **Por microrregião:** média entre as estações (média, e não soma, pelo mesmo motivo do pipeline).
3. **Anomalia padronizada (z):** para cada microrregião e **mês do calendário**, z = (chuva − média do mês) / desvio do mês. Isso remove a sazonalidade, porque "janeiro chuvoso" e "agosto chuvoso" não são comparáveis em mm.
4. Ficam só os meses da janela da safra (set–abr). Os meses secos têm desvio padrão perto de zero, o que gera anomalias instáveis.

In [ ]:
mapa = mapa_estacoes_cd_micro(PROCESSED / "clima_diario.parquet").dropna(subset=["CD_MICRO"])
if P["so_estacoes_no_poligono"]:
    mapa = mapa[mapa["origem"] == "dentro_do_poligono"]

clima["mes_ref"] = clima["data"].dt.to_period("M")
mensal = (clima.groupby(["codigo_estacao", "mes_ref"])
               .agg(chuva=("prec_total", "sum"), dias=("prec_total", "count")).reset_index())
mensal = mensal[mensal["dias"] >= P["completude_mensal"] * mensal["mes_ref"].dt.days_in_month]

chuva_micro = (mensal.merge(mapa[["codigo_estacao", "CD_MICRO"]], on="codigo_estacao")
                     .groupby(["CD_MICRO", "mes_ref"])["chuva"].mean().reset_index())
chuva_micro["mes"] = chuva_micro["mes_ref"].dt.month
g = chuva_micro.groupby(["CD_MICRO", "mes"])["chuva"]
chuva_micro["n_anos"] = g.transform("count")
chuva_micro["z"] = (chuva_micro["chuva"] - g.transform("mean")) / g.transform("std")
chuva_micro.loc[chuva_micro["n_anos"] < P["min_anos_climatologia"], "z"] = np.nan

anom = (chuva_micro[chuva_micro["mes"].isin(P["meses_safra"])]
        .pivot(index="mes_ref", columns="CD_MICRO", values="z").sort_index())
if P["so_micros_com_soja"]:
    anom = anom[[c for c in anom.columns if c in producao.index]]
anom = anom.dropna(axis=1, how="all")
print(f"Anomalias: {anom.shape[0]} meses de safra × {anom.shape[1]} microrregiões "
      f"| meses válidos por micro (mediana): {int(anom.notna().sum().median())}")

## 3. Correlação ENSO × chuva em cada microrregião (sem divisão)
- **Correlação de Spearman:** a chuva é assimétrica, e o posto é robusto a meses extremos.
- **Amostra efetiva:** o ONI é uma média móvel de 3 meses, muito autocorrelacionada, então os meses não são independentes. O p-valor usa a **amostra efetiva** de Bretherton et al. (1999): n_eff = n·(1 − ρ₁ˣρ₁ʸ)/(1 + ρ₁ˣρ₁ʸ).
- **Testes múltiplos:** são centenas de microrregiões, então os p-valores são corrigidos por **FDR (Benjamini–Hochberg)**.

In [ ]:
def serie_enso(nome: str, lag: int = 0) -> pd.Series:
    # Índice defasado: o valor no mês t é o do ENSO em t - lag
    return enso[nome].shift(lag)


def corr_neff(x: pd.Series, y: pd.Series, n_min: int = 24) -> tuple[float, float, float]:
    ok = x.notna() & y.notna()
    n = int(ok.sum())
    if n < n_min:
        return np.nan, np.nan, n
    xs, ys = x[ok], y[ok]
    r = stats.spearmanr(xs, ys).statistic
    a, b = xs.autocorr(1), ys.autocorr(1)
    a = 0.0 if np.isnan(a) else a
    b = 0.0 if np.isnan(b) else b
    n_eff = float(np.clip(n * (1 - a * b) / (1 + a * b), 3, n))
    t = r * np.sqrt((n_eff - 2) / max(1 - r ** 2, 1e-12))
    p = 2 * stats.t.sf(abs(t), df=n_eff - 2)
    return r, p, n_eff


def correlacao_por_micro(indice: str = "oni", lag: int = 0) -> pd.DataFrame:
    x = serie_enso(indice, lag).reindex(anom.index)
    linhas = [(c, *corr_neff(x, anom[c])) for c in anom.columns]
    df = pd.DataFrame(linhas, columns=["CD_MICRO", "r", "p", "n_eff"]).dropna(subset=["r"])
    df["p_fdr"] = multipletests(df["p"], method="fdr_bh")[1]
    df["sig"] = df["p_fdr"] < P["alpha_fdr"]
    return df


r_oni = correlacao_por_micro("oni", 0)
r_soi = correlacao_por_micro("soi", 0)
print(f"ONI: {r_oni['sig'].mean():.0%} das microrregiões com correlação significativa (FDR {P['alpha_fdr']})")
print(f"SOI: {r_soi['sig'].mean():.0%} | concordância ONI × SOI (r entre os mapas): "
      f"{r_oni.merge(r_soi, on='CD_MICRO')[['r_x', 'r_y']].corr().iloc[0, 1]:.2f}  (esperado: negativo)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 9))
for ax, (nome, df) in zip(axes, [("ONI (El Niño > 0)", r_oni), ("SOI (El Niño < 0)", r_soi)]):
    gdf = micro.merge(df, on="CD_MICRO", how="left")
    micro.plot(ax=ax, color="#eeeeee", edgecolor="white", linewidth=0.2)
    gdf.dropna(subset=["r"]).plot(ax=ax, column="r", cmap="BrBG", vmin=-0.4, vmax=0.4, legend=True,
                                  edgecolor="white", linewidth=0.2,
                                  legend_kwds={"label": "Spearman ρ (índice × anomalia de chuva)", "shrink": 0.6})
    gdf[gdf["sig"] == True].boundary.plot(ax=ax, color="black", linewidth=0.6)
    ax.set_title(f"{nome}, defasagem 0 · contorno preto = significativo (FDR)")
    ax.axis("off")
plt.tight_layout()
plt.show()

**Como ler:** com o ONI, verde (ρ > 0) indica mais chuva em El Niño, e marrom (ρ < 0) indica menos chuva. O padrão esperado pela literatura é Sul **verde** e Norte/Nordeste **marrom**. O mapa do SOI deve ser o espelho (sinal invertido).

## 4. Divisões regionais candidatas
Cada microrregião recebe **um** rótulo por divisão. Nas camadas que não coincidem com os limites das microrregiões (biomas e regiões intermediárias), vale **a de maior área de interseção**, calculada numa projeção de áreas.

In [ ]:
def atribuir_por_maior_area(camada: gpd.GeoDataFrame, coluna: str) -> pd.Series:
    m = micro[["CD_MICRO", "geometry"]].to_crs(P["crs_area"])
    c = camada[[coluna, "geometry"]].to_crs(P["crs_area"])
    m["geometry"], c["geometry"] = m.geometry.make_valid(), c.geometry.make_valid()
    inter = gpd.overlay(m, c, how="intersection", keep_geom_type=True)
    inter["area"] = inter.area
    return inter.sort_values("area").drop_duplicates("CD_MICRO", keep="last").set_index("CD_MICRO")[coluna]


micro_idx = micro.set_index("CD_MICRO")
divisoes: dict[str, pd.Series] = {
    "Macrorregião (5)": micro_idx["name_region"],
    "UF (27)": micro_idx["abbrev_state"],
}

biomas = geobr.read_biomes(year=2019)
biomas = biomas[~biomas["name_biome"].str.contains("Costeiro", case=False, na=False)]
divisoes["Bioma (IBGE)"] = atribuir_por_maior_area(biomas, "name_biome")

intermed = geobr.read_intermediate_region(year=P["ano_malha"])
col_int = "name_intermediate" if "name_intermediate" in intermed.columns else intermed.columns[1]
intermed["rotulo"] = intermed[col_int].astype(str) + " (" + intermed["abbrev_state"].astype(str) + ")"
divisoes["Região intermediária (IBGE)"] = atribuir_por_maior_area(intermed, "rotulo")

# MATOPIBA (APROXIMAÇÃO): micros de MA/TO/PI/BA com predomínio de Cerrado.
# A delimitação oficial (Embrapa/GITE, 2015) também parte de microrregiões com Cerrado
# nesses estados (31 microrregiões); compare a contagem abaixo com essa referência.
matopiba = (micro_idx["abbrev_state"].isin(["MA", "TO", "PI", "BA"])
            & (divisoes["Bioma (IBGE)"].reindex(micro_idx.index) == "Cerrado"))
print(f"MATOPIBA aproximado: {matopiba.sum()} microrregiões (referência oficial: 31) | "
      f"{producao.reindex(matopiba[matopiba].index).sum() / producao.sum():.1%} da produção média da base")
divisoes["Macrorregião + MATOPIBA"] = micro_idx["name_region"].where(~matopiba, "MATOPIBA")

# Variante: UF, com as UFs que têm menos de `min_micros_regiao` microrregiões com dados
# agrupadas no "resto" da sua macrorregião (evita regiões de 1-2 microrregiões)
n_por_uf = micro_idx["abbrev_state"].reindex(anom.columns).value_counts()
ufs_pequenas = n_por_uf[n_por_uf < P["min_micros_regiao"]].index
divisoes["UF (pequenas → macrorregião)"] = micro_idx["abbrev_state"].where(
    ~micro_idx["abbrev_state"].isin(ufs_pequenas), "resto " + micro_idx["name_region"])
print(f"UFs agrupadas por terem < {P['min_micros_regiao']} microrregiões com dados: {sorted(ufs_pequenas)}")

# Variante: Bioma com o Pantanal junto ao Cerrado (vizinho e de clima semelhante)
n_por_bioma = divisoes["Bioma (IBGE)"].reindex(anom.columns).value_counts()
print("Microrregiões com dados por bioma:", n_por_bioma.to_dict())
divisoes["Bioma (Pantanal → Cerrado)"] = divisoes["Bioma (IBGE)"].replace({"Pantanal": "Cerrado"})

# Regiões edafoclimáticas da soja (Embrapa): arquivo externo CD_MICRO,regiao
arq_rec = RAW / "regioes" / "rec_soja.csv"
if arq_rec.exists():
    rec = pd.read_csv(arq_rec, dtype={"CD_MICRO": str}).set_index("CD_MICRO")["regiao"]
    divisoes["Regiões edafoclimáticas da soja (Embrapa)"] = rec
else:
    print(f"⚠️ {arq_rec} não encontrado: regiões da Embrapa NÃO avaliadas "
          "(colunas esperadas: CD_MICRO, regiao).")

for nome, rot in divisoes.items():
    print(f"{nome:45s} {rot.nunique():4d} regiões")

## 5. Avaliação dentro da amostra (descritiva)
> ⚠️ Estas métricas usam os mesmos anos em que as correlações foram estimadas. Servem para **descrever** as divisões, não para **escolher** entre elas: divisões com muitas regiões pequenas se ajustam ao ruído e parecem melhores do que são. A escolha usa a seção 6.

Para cada divisão e cada região:
- **série regional:** média das anomalias (z) das microrregiões da região, mês a mês;
- **r regional:** correlação de Spearman com o ENSO (amostra efetiva e FDR dentro da divisão).

Métricas por divisão (↑ = melhor):

| Métrica | O que mede |
|---|---|
| `homogeneidade_R2adj` ↑ | fração da variação do ρ entre microrregiões explicada pela divisão, penalizada pelo nº de regiões |
| `coerencia_sinal` ↑ | % de microrregiões com o mesmo sinal da mediana da sua região |
| `forca_regional` ↑ | média de \|r regional\|, ponderada pela produção de soja |
| `prod_em_regioes_sig` ↑ | % da produção de soja em regiões com correlação significativa |
| `instabilidade` ↓ | diferença média \|r(safras pares) − r(safras ímpares)\| entre as regiões, ponderada pela produção |
| `R2adj_SOI`, `R2adj_lag3` ↑ | homogeneidade com o SOI e com defasagem de 3 meses (robustez) |
| `pct_micros_regioes_frageis` ↓ | % de microrregiões em regiões com menos de `min_micros_regiao` membros |

In [ ]:
ano_safra = pd.Series(np.where(anom.index.month >= 9, anom.index.year + 1, anom.index.year), index=anom.index)


def r2_ajustado(valores: pd.Series, grupos: pd.Series) -> float:
    df = pd.DataFrame({"v": valores, "g": grupos}).dropna()
    n, k = len(df), df["g"].nunique()
    if k < 2 or n <= k:
        return np.nan
    sst = ((df["v"] - df["v"].mean()) ** 2).sum()
    ssw = ((df["v"] - df.groupby("g")["v"].transform("mean")) ** 2).sum()
    return 1 - (ssw / (n - k)) / (sst / (n - 1))


def series_regionais(rotulos: pd.Series) -> pd.DataFrame:
    rot = rotulos.reindex(anom.columns).dropna()
    return pd.DataFrame({reg: anom[mics].mean(axis=1) for reg, mics in rot.groupby(rot).groups.items()})


def tabela_regional(rotulos: pd.Series, indice: str = "oni", lag: int = 0, mascara=None) -> pd.DataFrame:
    reg = series_regionais(rotulos)
    if mascara is not None:
        reg = reg[mascara]
    x = serie_enso(indice, lag).reindex(reg.index)
    linhas = [(r, *corr_neff(x, reg[r])) for r in reg.columns]
    t = pd.DataFrame(linhas, columns=["regiao", "r", "p", "n_eff"]).dropna(subset=["r"])
    if len(t):
        t["p_fdr"] = multipletests(t["p"], method="fdr_bh")[1]
        t["sig"] = t["p_fdr"] < P["alpha_fdr"]
    rot = rotulos.reindex(anom.columns).dropna()
    t["n_micros"] = t["regiao"].map(rot.value_counts())
    t["producao"] = t["regiao"].map(producao.reindex(rot.index).groupby(rot).sum())
    return t.sort_values("r")


def avaliar(nome: str, rotulos: pd.Series) -> tuple[dict, pd.DataFrame]:
    rmi = r_oni.set_index("CD_MICRO")
    rot = rotulos.reindex(rmi.index)
    reg = tabela_regional(rotulos, "oni", 0)

    mediana = rmi["r"].groupby(rot).transform("median")
    coerencia = (np.sign(rmi["r"]) == np.sign(mediana))[rot.notna()].mean()

    w = reg["producao"].fillna(0)
    forca = np.average(reg["r"].abs(), weights=w) if w.sum() > 0 else np.nan
    prod_sig = w[reg["sig"]].sum() / w.sum() if w.sum() > 0 else np.nan

    # Estabilidade: |r(pares) - r(ímpares)| por região, ponderado pela produção.
    # (Correlacionar os r entre as metades não serve com 5-6 regiões: poucos pontos.)
    par = tabela_regional(rotulos, "oni", 0, mascara=(ano_safra % 2 == 0).values).set_index("regiao")
    imp = tabela_regional(rotulos, "oni", 0, mascara=(ano_safra % 2 == 1).values).set_index("regiao")
    comum = par.index.intersection(imp.index)
    dif = (par.loc[comum, "r"] - imp.loc[comum, "r"]).abs()
    pesos = par.loc[comum, "producao"].fillna(0)
    instab = np.average(dif, weights=pesos) if pesos.sum() > 0 else np.nan

    r_lag = correlacao_por_micro("oni", P["lag_robustez"]).set_index("CD_MICRO")["r"]
    tamanhos = rot.value_counts()
    frageis = rot.map(tamanhos).lt(P["min_micros_regiao"])[rot.notna()].mean()

    m = dict(
        divisao=nome,
        n_regioes=int(rot.nunique()),
        micros_por_regiao_mediana=float(tamanhos.median()),
        homogeneidade_R2adj=r2_ajustado(rmi["r"], rot),
        coerencia_sinal=coerencia,
        forca_regional=forca,
        prod_em_regioes_sig=prod_sig,
        instabilidade=instab,
        R2adj_SOI=r2_ajustado(r_soi.set_index("CD_MICRO")["r"], rotulos.reindex(r_soi["CD_MICRO"]).set_axis(r_soi["CD_MICRO"])),
        R2adj_lag3=r2_ajustado(r_lag, rotulos.reindex(r_lag.index)),
        pct_micros_regioes_frageis=frageis,
    )
    return m, reg


metricas, regionais = [], {}
for nome, rot in divisoes.items():
    m, reg = avaliar(nome, rot)
    metricas.append(m)
    regionais[nome] = reg

tabela = pd.DataFrame(metricas).set_index("divisao")
criterios = ["homogeneidade_R2adj", "coerencia_sinal", "forca_regional", "prod_em_regioes_sig",
             "R2adj_SOI", "R2adj_lag3"]                       # maior = melhor
criterios_menor = ["instabilidade", "pct_micros_regioes_frageis"]  # menor = melhor
ranks = tabela[criterios].rank(ascending=False)
for c in criterios_menor:
    ranks[c] = tabela[c].rank(ascending=True)
tabela["rank_medio"] = ranks.mean(axis=1)
tabela = tabela.sort_values("rank_medio")
(tabela.style.format(precision=2)
       .background_gradient(subset=criterios, cmap="Greens")
       .background_gradient(subset=criterios_menor, cmap="Greens_r"))

O **rank médio** desta tabela é só descritivo. Na primeira execução, ele colocou a Região intermediária e a UF no topo, e as duas pioraram fora da amostra (seção 6), em especial a Região intermediária, cuja mediana é de 2 microrregiões por região.

In [ ]:
todos = criterios + criterios_menor
fig, axes = plt.subplots(1, len(todos), figsize=(3.6 * len(todos), 4), sharey=True)
for ax, c in zip(axes, todos):
    tabela[c].plot.barh(ax=ax, color="#4c8c4a")
    ax.set_title(c + (" (↓ melhor)" if c in criterios_menor else ""), fontsize=10)
    ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Avaliação fora da amostra (critério principal)
**Teste:**
1. calcula-se o ρ (ENSO × chuva) de cada microrregião usando só as safras **pares** e, separadamente, só as **ímpares**;
2. a previsão para uma microrregião é a **média do ρ das outras microrregiões da sua região** no conjunto de treino (a própria microrregião fica de fora);
3. a previsão é comparada com o ρ da microrregião no conjunto de teste, nos dois sentidos (pares → ímpares e ímpares → pares).

**Métricas:**

| Métrica | O que mede |
|---|---|
| `R2_fora` ↑ | quanto a média da região prevê a microrregião em anos que não foram usados; a referência é **"Sem divisão"** (média nacional, R² ≈ 0) |
| `erro_pond` ↓ | erro da previsão, ponderado pela produção de soja (pesa mais onde há soja) |
| `repet_regional` ↑ | o ρ **regional** dos anos pares se repete nos ímpares? (correlação entre regiões; ruidosa com poucas regiões) |

A **repetibilidade microrregional** (correlação do ρ de cada microrregião entre as duas metades) é o teto do que qualquer divisão consegue prever.

In [ ]:
def r_micro_mascara(mascara, indice: str = "oni", lag: int = 0) -> pd.Series:
    x = serie_enso(indice, lag).reindex(anom.index)[mascara]
    return pd.Series({c: corr_neff(x, anom[c][mascara], n_min=20)[0] for c in anom.columns}).dropna()


def prever_pela_regiao(treino: pd.Series, teste: pd.Series, rotulos: pd.Series) -> tuple[float, float]:
    comum = treino.index.intersection(teste.index)
    lab = rotulos.reindex(comum)
    ok = lab.notna()
    tr, te, lab = treino[comum][ok], teste[comum][ok], lab[ok]
    soma, n = tr.groupby(lab).transform("sum"), tr.groupby(lab).transform("count")
    pred = ((soma - tr) / (n - 1)).where(n > 1, tr.mean())   # média da região sem a própria micro
    r2 = 1 - ((te - pred) ** 2).sum() / ((te - te.mean()) ** 2).sum()
    pesos = producao.reindex(te.index).fillna(0) + 1e-9
    return r2, float(np.sqrt(np.average((te - pred) ** 2, weights=pesos)))


par, imp = (ano_safra % 2 == 0).values, (ano_safra % 2 == 1).values
divs_oos = {"Sem divisão (Brasil)": pd.Series("Brasil", index=micro_idx.index), **divisoes}
linhas, repetibilidade_micro = [], {}
for lag in sorted({0, P["lag_robustez"]}):
    rp, ri = r_micro_mascara(par, lag=lag), r_micro_mascara(imp, lag=lag)
    comum = rp.index.intersection(ri.index)
    repetibilidade_micro[lag] = np.corrcoef(rp[comum], ri[comum])[0, 1]
    for nome, rot in divs_oos.items():
        a1, e1 = prever_pela_regiao(rp, ri, rot)
        a2, e2 = prever_pela_regiao(ri, rp, rot)
        rr = np.nan
        if nome != "Sem divisão (Brasil)":
            tp = tabela_regional(rot, "oni", lag, mascara=par).set_index("regiao")["r"]
            ti = tabela_regional(rot, "oni", lag, mascara=imp).set_index("regiao")["r"]
            c = tp.index.intersection(ti.index)
            rr = np.corrcoef(tp[c], ti[c])[0, 1] if len(c) >= 3 else np.nan
        linhas.append(dict(divisao=nome, lag=lag, R2_fora=(a1 + a2) / 2, erro_pond=(e1 + e2) / 2, repet_regional=rr))

oos = pd.DataFrame(linhas).pivot(index="divisao", columns="lag", values=["R2_fora", "erro_pond", "repet_regional"])
oos.columns = [f"{m}_lag{l}" for m, l in oos.columns]
oos.insert(0, "n_regioes", [1 if d == "Sem divisão (Brasil)" else int(divisoes[d].reindex(anom.columns).nunique()) for d in oos.index])
cols_r2 = [c for c in oos.columns if c.startswith("R2_fora")]
oos["R2_fora_medio"] = oos[cols_r2].mean(axis=1)
oos = oos.sort_values("R2_fora_medio", ascending=False)

for lag, rep_ in repetibilidade_micro.items():
    print(f"Repetibilidade do ρ microrregional entre metades (defasagem {lag}): {rep_:.2f}  "
          f"→ estimada para a série inteira (Spearman-Brown): {2 * rep_ / (1 + rep_):.2f}")
(oos.style.format(precision=3)
    .background_gradient(subset=cols_r2 + ["R2_fora_medio"], cmap="Greens")
    .background_gradient(subset=[c for c in oos.columns if c.startswith("erro_pond")], cmap="Greens_r"))

**Como ler:**
- **R² fora da amostra negativo** significa que a média da região prevê pior que a média nacional. Isso acontece quando o ρ de cada microrregião é dominado por ruído: a média de uma região pequena é tão ruidosa quanto a microrregião isolada.
- Uma **repetibilidade microrregional baixa** (≈ 0,2) indica que, com estes dados e sem defasagem, a resposta ao ENSO **no nível da microrregião** quase não se repete entre períodos. O sinal confiável é o **agregado** (regiões grandes).
- Entre as divisões, prefira a que tem **maior R² fora da amostra e menor erro ponderado nas duas defasagens**, com **poucas regiões** (para a varredura de defasagens não multiplicar os testes).

## 7. Mapas das divisões: r regional (ONI, defasagem 0)
A cor é o ρ da série regional com o ONI. Regiões **hachuradas** não são significativas (FDR); as cinzas não têm microrregião com soja e dados.

In [ ]:
n = len(divisoes)
ncol = 3
fig, axes = plt.subplots(int(np.ceil(n / ncol)), ncol, figsize=(7 * ncol, 7 * int(np.ceil(n / ncol))))
for ax, (nome, rot) in zip(np.ravel(axes), divisoes.items()):
    reg = regionais[nome].set_index("regiao")
    g = micro.assign(regiao=micro["CD_MICRO"].map(rot)).dissolve(by="regiao").reset_index()
    g = g.merge(reg[["r", "sig"]], left_on="regiao", right_index=True, how="left")
    micro.plot(ax=ax, color="#eeeeee", linewidth=0)
    g.dropna(subset=["r"]).plot(ax=ax, column="r", cmap="BrBG", vmin=-0.4, vmax=0.4, edgecolor="black", linewidth=0.3)
    nsig = g[g["sig"] == False]
    if len(nsig):
        nsig.plot(ax=ax, facecolor="none", hatch="///", edgecolor="grey", linewidth=0)
    ax.set_title(f"{nome}\nR² fora da amostra = {oos.loc[nome, 'R2_fora_lag0']:.2f} · R²aj (dentro) = {tabela.loc[nome, 'homogeneidade_R2adj']:.2f}")
    ax.axis("off")
for ax in np.ravel(axes)[n:]:
    ax.axis("off")
sm = plt.cm.ScalarMappable(cmap="BrBG", norm=plt.Normalize(-0.4, 0.4))
fig.colorbar(sm, ax=list(np.ravel(axes)), shrink=0.4, label="ρ regional (ONI × anomalia de chuva)")
plt.show()

In [ ]:
# Detalhe por região da divisão mais bem colocada FORA da amostra (troque o nome para inspecionar outra)
escolhida = oos.drop(index="Sem divisão (Brasil)").index[0]
print(f"Divisão: {escolhida}")
regionais[escolhida].round(3)

## 8. Decisão (registrada em `docs/decisoes.md`, D-01)
**Divisão provisória: Bioma, com o Pantanal agrupado ao Cerrado. A UF entra como análise de sensibilidade.**
- **Motivo:** entre as divisões com poucas regiões, o Bioma tem o melhor desempenho fora da amostra nas duas defasagens, o menor erro ponderado pela produção e o sinal regional mais repetível. Também tem motivação física (regimes de clima) e só ~5 regiões, o que mantém controlável o número de testes na varredura de defasagens.
- **UF:** empata com o Bioma sem defasagem, mas piora com 3 meses e tem UFs com 1–2 microrregiões. Fica como sensibilidade e para comunicação.
- **Descartadas:**
  - Região intermediária (pior fora da amostra: regiões pequenas demais);
  - Macrorregião + MATOPIBA (separar o MATOPIBA não melhora);
  - Macrorregião (sempre abaixo do Bioma).
- **Ressalva:** nenhuma divisão supera a média nacional fora da amostra, e as diferenças entre Bioma, UF e Macrorregião são pequenas. O que os dados sustentam com segurança é a gangorra Sul (+) × Norte (−) em escala grande.
- **Reavaliação com defasagens (seção 14):** a escolha **se mantém**. Com defasagens de 0, 3, 6 e 9 meses, o Bioma (Pantanal → Cerrado) tem o melhor R² fora da amostra médio e a melhor posição média entre as divisões.

## 9. Próximos passos
1. **Defasagens**: Parte 2, abaixo.
2. **Reavaliar a divisão** com as defasagens escolhidas.
3. Repetir com a anomalia de **rendimento** (requer o `desvio_clima` por microrregião) e incorporar ao pipeline.

**Limitações:**
- o MATOPIBA é aproximado;
- as regiões da Embrapa dependem de um arquivo externo;
- a amostra efetiva é uma aproximação;
- cada metade da validação fora da amostra tem só cerca de 10 safras.

---
# Parte 2 · Defasagens entre o ENSO e a chuva

**Pergunta:** com quantos meses de atraso a chuva de cada região mais se relaciona com o ENSO? E esse padrão se repete em outro período?

Esta é uma **análise de dados**, não um modelo de previsão. Para cada mês de chuva *t*, o valor do ENSO comparado é o **índice ENSO defasado**: a média do índice nos *w* meses que terminam *L* meses antes de *t*.
- **Defasagem L:** de 0 a 12 meses.
- **Janela w:** 1, 3 ou 6 meses. O ONI já é uma média de 3 meses, então w = 1 com o ONI já cobre 3 meses.
- **Exemplo:** para a chuva de janeiro de 2015, L = 6 e w = 3 comparam com a média do ONI de maio a julho de 2014.

**Dois períodos:**
- **Exploração (safras até 2020):** os padrões são procurados aqui.
- **Verificação de robustez (safras de 2021 a 2025):** os padrões encontrados são verificados **uma única vez** neste período, que não foi usado para encontrá-los. Os critérios da verificação foram fixados (seção 12) **antes** de olhar esse período.

Decisões registradas em `docs/decisoes.md` (D-02 e D-03).

In [ ]:
P_LAG = dict(
    lags=list(range(0, 13)),
    janelas=[1, 3, 6],
    safra_ultima_exploracao=2020,       # padrões procurados nas safras <= 2020
    safras_verificacao=(2021, 2025),    # verificação de robustez, feita uma única vez
    divisao_principal="Bioma (Pantanal → Cerrado)",
    divisao_sensibilidade="UF (pequenas → macrorregião)",
    # critérios de consistência (opção 2, decisão D-02)
    plato_fracao=2 / 3,                 # vizinhas (L-1, L+1) com |ρ| médio >= 2/3 do pico, mesmo sinal
    soi_tolerancia_meses=2,             # defasagem do SOI a até 2 meses da do ONI, com sinal oposto
    # critério da verificação 2021–2025 (fixado antes de olhar esse período)
    rho_min_verificacao=0.10,           # "se repete" = mesmo sinal e |ρ| >= 0,10
    n_bootstrap=2000,
)

explora = (ano_safra <= P_LAG["safra_ultima_exploracao"]).values
verifica = ano_safra.between(*P_LAG["safras_verificacao"]).values
print(f"Exploração: safras {int(ano_safra[explora].min())}–{int(ano_safra[explora].max())} ({explora.sum()} meses) | "
      f"verificação: {P_LAG['safras_verificacao'][0]}–{P_LAG['safras_verificacao'][1]} ({verifica.sum()} meses)")

# Índice ENSO combinado (opção 1): média de z(ONI) e -z(SOI) no período em que ambos existem
_comum = enso[["oni", "soi"]].dropna()
_z = (_comum - _comum.mean()) / _comum.std()
enso["comb"] = ((_z["oni"] - _z["soi"]) / 2).reindex(enso.index)
print(f"Correlação mensal ONI × SOI: {_comum['oni'].corr(_comum['soi']):.2f}")


def enso_janela(indice: str, lag: int, janela: int) -> pd.Series:
    # média do índice nos `janela` meses que terminam `lag` meses antes de t
    return enso[indice].rolling(janela, min_periods=janela).mean().shift(lag)


def varredura(rotulos: pd.Series, indice: str, mascara) -> pd.DataFrame:
    reg = series_regionais(rotulos)[mascara]
    linhas = []
    for w in P_LAG["janelas"]:
        for L in P_LAG["lags"]:
            x = enso_janela(indice, L, w).reindex(reg.index)
            for r in reg.columns:
                rho, p, n_eff = corr_neff(x, reg[r])
                linhas.append(dict(regiao=r, indice=indice, lag=L, janela=w, rho=rho, p=p, n_eff=n_eff))
    df = pd.DataFrame(linhas).dropna(subset=["rho"])
    df["p_fdr"] = multipletests(df["p"], method="fdr_bh")[1]
    df["sig"] = df["p_fdr"] < P["alpha_fdr"]
    df = df.sort_values(["regiao", "janela", "lag"])
    df["abs_rho_suav"] = (df.groupby(["regiao", "janela"])["rho"]
                            .transform(lambda v: v.abs().rolling(3, center=True, min_periods=2).mean()))
    return df


varreduras = {}
for nome in (P_LAG["divisao_principal"], P_LAG["divisao_sensibilidade"]):
    for indice in ("oni", "soi", "comb"):
        v = varredura(divisoes[nome], indice, explora)
        varreduras[(nome, indice)] = v
        print(f"{nome:32s} {indice.upper():4s}: {len(v)} combinações | {v['sig'].sum()} significativas após FDR")

## 10. Mapas de calor: região × defasagem (exploração, safras até 2020)
Uma linha por região e um painel por janela. A cor é o ρ entre o índice defasado e a anomalia de chuva; **\*** marca as combinações significativas após o FDR. Com o ONI, verde indica mais chuva em El Niño e marrom indica menos. O SOI tem sinal oposto, então o mapa dele deve ser o espelho.

In [ ]:
def heatmap_varredura(nome: str, indice: str = "oni") -> None:
    v = varreduras[(nome, indice)]
    ordem = v.groupby("regiao")["rho"].mean().sort_values().index
    fig, axes = plt.subplots(1, len(P_LAG["janelas"]), sharey=True, layout="constrained",
                             figsize=(6 * len(P_LAG["janelas"]), 0.42 * len(ordem) + 1.8))
    for ax, w in zip(np.ravel([axes]), P_LAG["janelas"]):
        m = v[v["janela"] == w].pivot(index="regiao", columns="lag", values="rho").reindex(ordem)
        sig = v[v["janela"] == w].pivot(index="regiao", columns="lag", values="sig").reindex(ordem)
        im = ax.imshow(m.values, cmap="BrBG", vmin=-0.4, vmax=0.4, aspect="auto")
        for (i, j), val in np.ndenumerate(sig.values):
            if val is True or val == 1:
                ax.text(j, i, "*", ha="center", va="center", fontsize=12)
        ax.set_xticks(range(len(m.columns)), m.columns)
        ax.set_yticks(range(len(m.index)), m.index)
        ax.set_xlabel("defasagem (meses antes do mês da chuva)")
        ax.set_title(f"janela de {w} mês(es)")
    fig.colorbar(im, ax=list(np.ravel([axes])), shrink=0.8, label=f"ρ ({indice.upper()} defasado × anomalia de chuva)")
    fig.suptitle(f"{nome} · {indice.upper()} · exploração (safras até {P_LAG['safra_ultima_exploracao']})")
    plt.show()


heatmap_varredura(P_LAG["divisao_principal"], "oni")
heatmap_varredura(P_LAG["divisao_principal"], "soi")

## 11. Como identificar uma defasagem: comparação de três critérios (exploração)
Com 195 combinações por índice, muito correlacionadas entre si (defasagens vizinhas e janelas sobrepostas medem quase o mesmo), a correção FDR fica muito conservadora. Por transparência, os três critérios considerados são comparados abaixo, **todos só com as safras até 2020**:

| Opção | Critério |
|---|---|
| 3 · ONI + FDR | combinação do ONI significativa após o FDR (critério original) |
| 1 · índice combinado + FDR | o mesmo, com o índice z(ONI) − z(SOI) |
| 2 · ONI + consistência | maior \|ρ\| suavizado do ONI, exigindo: **platô** (defasagens vizinhas com o mesmo sinal e ao menos 2/3 do pico), **SOI concordante** (defasagem a até 2 meses, sinal oposto) e **estabilidade** (mesmo sinal nas safras pares e ímpares) |

**Decisão (D-02):** a opção 2 aponta os **padrões candidatos**, e as opções 1 e 3 marcam os casos de **evidência mais forte**. A confirmação vem do período independente (seção 12).

In [ ]:
def escolher_fdr(v: pd.DataFrame) -> dict:
    out = {}
    for reg, g in v.groupby("regiao"):
        s = g[g["sig"]]
        out[reg] = None if s.empty else s.loc[s["abs_rho_suav"].idxmax()]
    return out


def escolher_consistencia(nome: str) -> dict:
    v_oni, v_soi = varreduras[(nome, "oni")], varreduras[(nome, "soi")]
    serie_reg = series_regionais(divisoes[nome])
    out = {}
    for reg, g in v_oni.groupby("regiao"):
        best = g.loc[g["abs_rho_suav"].idxmax()]
        L, w, rho = int(best["lag"]), int(best["janela"]), best["rho"]
        viz = g[(g["janela"] == w) & g["lag"].between(L - 1, L + 1)]
        plato = bool((np.sign(viz["rho"]) == np.sign(rho)).all()
                     and viz["rho"].abs().mean() >= P_LAG["plato_fracao"] * abs(rho))
        gs = v_soi[v_soi["regiao"] == reg]
        bs = gs.loc[gs["abs_rho_suav"].idxmax()]
        soi_ok = bool(abs(int(bs["lag"]) - L) <= P_LAG["soi_tolerancia_meses"] and np.sign(bs["rho"]) == -np.sign(rho))
        x = enso_janela("oni", L, w).reindex(anom.index)
        r_par = corr_neff(x[explora & par], serie_reg[reg][explora & par], 12)[0]
        r_imp = corr_neff(x[explora & imp], serie_reg[reg][explora & imp], 12)[0]
        estavel = bool(np.sign(r_par) == np.sign(r_imp) == np.sign(rho))
        out[reg] = dict(lag=L, janela=w, rho=rho, plato=plato, soi_ok=soi_ok, lag_soi=int(bs["lag"]),
                        estavel=estavel, rho_pares=r_par, rho_impares=r_imp, candidato=plato and soi_ok and estavel)
    return out


def comparar_criterios(nome: str) -> pd.DataFrame:
    e3, e1, e2 = (escolher_fdr(varreduras[(nome, "oni")]), escolher_fdr(varreduras[(nome, "comb")]),
                  escolher_consistencia(nome))
    rot = divisoes[nome].reindex(anom.columns).dropna()
    prod = producao.reindex(rot.index).groupby(rot).sum() / 1e6
    fmt = lambda b: "—" if b is None else f"{int(b['lag'])} m (j{int(b['janela'])}) ρ={b['rho']:+.2f}"
    linhas = []
    for reg, c in sorted(e2.items(), key=lambda kv: kv[1]["rho"]):
        falhas = [k for k in ("plato", "soi_ok", "estavel") if not c[k]]
        linhas.append({"região": reg, "soja (Mt)": round(prod.get(reg, np.nan), 1),
                       "opção 3 · ONI+FDR": fmt(e3.get(reg)), "opção 1 · combinado+FDR": fmt(e1.get(reg)),
                       "opção 2 · ONI+consistência": (f"✓ {c['lag']} m (j{c['janela']}) ρ={c['rho']:+.2f}" if c["candidato"]
                                                     else f"✗ ({', '.join(falhas)})")})
    return pd.DataFrame(linhas)


comparar_criterios(P_LAG["divisao_principal"])

## 12. Verificação de robustez nas safras de 2021 a 2025
**Critérios fixados antes de olhar este período:**
- cada candidato é verificado com a **mesma** defasagem e janela encontradas na exploração, com o ONI;
- **"se repete"** = mesmo sinal da exploração **e** |ρ| ≥ 0,10 nas safras de 2021 a 2025;
- **níveis de evidência:**

| Nível | Critério |
|---|---|
| **forte** | candidato que **também** passa no FDR (opção 1 ou 3) **e** se repete |
| **moderada** | candidato (opção 2) que se repete |
| **não confirmada** | candidato que não se repete |
| **sem padrão consistente** | não é candidato (falhou no platô, no SOI ou na estabilidade) |

São só 5 safras (cerca de 40 meses), então o intervalo de confiança de 95% (bootstrap por safra) é largo. A verificação diz se o padrão **se mantém**, não se ele é estatisticamente significativo nesse período curto.

In [ ]:
def bootstrap_por_safra(x: pd.Series, y: pd.Series, safras: pd.Series, n: int, seed: int = 42) -> tuple[float, float]:
    rng = np.random.default_rng(seed)
    unicas = safras.unique()
    vals = []
    for _ in range(n):
        amostra = rng.choice(unicas, size=len(unicas), replace=True)
        idx = np.concatenate([np.where(safras.values == a)[0] for a in amostra])
        xs, ys = x.iloc[idx], y.iloc[idx]
        ok = xs.notna() & ys.notna()
        if ok.sum() >= 10 and xs[ok].nunique() > 1:
            vals.append(stats.spearmanr(xs[ok], ys[ok]).statistic)
    return tuple(np.nanpercentile(vals, [2.5, 97.5])) if vals else (np.nan, np.nan)


def verificar(nome: str) -> pd.DataFrame:
    cand = escolher_consistencia(nome)
    fdr3, fdr1 = escolher_fdr(varreduras[(nome, "oni")]), escolher_fdr(varreduras[(nome, "comb")])
    serie_reg = series_regionais(divisoes[nome])
    saf_v = ano_safra[verifica]
    rot = divisoes[nome].reindex(anom.columns).dropna()
    prod = producao.reindex(rot.index).groupby(rot).sum() / 1e6
    linhas = []
    for reg, c in cand.items():
        linha = {"região": reg, "soja (Mt)": round(prod.get(reg, np.nan), 1), "defasagem (meses)": c["lag"],
                 "janela": c["janela"], "ρ exploração": c["rho"], "lag SOI": c["lag_soi"]}
        if not c["candidato"]:
            linhas.append({**linha, "nível": "sem padrão consistente"})
            continue
        x = enso_janela("oni", c["lag"], c["janela"]).reindex(anom.index)
        y = serie_reg[reg]
        r_v = corr_neff(x[verifica], y[verifica], 12)[0]
        lo, hi = bootstrap_por_safra(x[verifica], y[verifica], saf_v, P_LAG["n_bootstrap"])
        r0_v = corr_neff(enso_janela("oni", 0, 1).reindex(anom.index)[verifica], y[verifica], 12)[0]
        repete = bool(np.sign(r_v) == np.sign(c["rho"]) and abs(r_v) >= P_LAG["rho_min_verificacao"])
        forte = fdr3.get(reg) is not None or fdr1.get(reg) is not None
        nivel = ("forte" if forte else "moderada") if repete else "não confirmada"
        linhas.append({**linha, "ρ 2021–2025": r_v, "IC95% 2021–2025": f"[{lo:+.2f}, {hi:+.2f}]",
                       "ρ 2021–2025 sem defasagem": r0_v, "passa no FDR": forte, "nível": nivel})
    ordem = {"forte": 0, "moderada": 1, "não confirmada": 2, "sem padrão consistente": 3}
    return pd.DataFrame(linhas).sort_values(["nível", "soja (Mt)"], key=lambda s: s.map(ordem) if s.name == "nível" else -s)


resultado_bioma = verificar(P_LAG["divisao_principal"])
resultado_bioma.round(2)

## 13. Sensibilidade: o mesmo procedimento por UF
Se um padrão de bioma for real, as UFs dentro dele devem mostrar defasagens parecidas.

In [ ]:
heatmap_varredura(P_LAG["divisao_sensibilidade"], "oni")
resultado_uf = verificar(P_LAG["divisao_sensibilidade"])
# bioma predominante de cada UF, pela PRODUÇÃO de soja (não pelo número de microrregiões)
_ub = (pd.DataFrame({"uf": divisoes[P_LAG["divisao_sensibilidade"]], "bioma": divisoes[P_LAG["divisao_principal"]],
                     "prod": producao}).reindex(anom.columns).dropna(subset=["uf", "bioma"]).fillna({"prod": 0}))
bioma_da_uf = _ub.groupby(["uf", "bioma"])["prod"].sum().reset_index().sort_values("prod").drop_duplicates("uf", keep="last").set_index("uf")["bioma"]
resultado_uf.insert(1, "bioma predominante", resultado_uf["região"].map(bioma_da_uf))
resultado_uf.round(2)

## 14. Reavaliação da divisão regional com defasagens (D-01)
O D-01 escolheu o Bioma sem defasagem e com 3 meses. Aqui a validação fora da amostra da seção 6 é repetida **só com as safras de exploração** (pares → ímpares e vice-versa), nas defasagens 0, 3, 6 e 9 meses (janela de 1 mês). A pergunta é se o Bioma continua entre as melhores divisões quando a defasagem muda.

In [ ]:
def r_micro_lag(mascara, lag: int) -> pd.Series:
    x = enso_janela("oni", lag, 1).reindex(anom.index)[mascara]
    return pd.Series({c: corr_neff(x, anom[c][mascara], n_min=20)[0] for c in anom.columns}).dropna()


lin = []
for lag in (0, 3, 6, 9):
    rp, ri = r_micro_lag(explora & par, lag), r_micro_lag(explora & imp, lag)
    for nome, rot in divs_oos.items():
        a1, e1 = prever_pela_regiao(rp, ri, rot)
        a2, e2 = prever_pela_regiao(ri, rp, rot)
        lin.append(dict(divisao=nome, lag=lag, R2_fora=(a1 + a2) / 2))
reav = pd.DataFrame(lin).pivot(index="divisao", columns="lag", values="R2_fora")
reav.columns = [f"R2_fora_lag{c}" for c in reav.columns]
reav["media"] = reav.mean(axis=1)
reav["posicao_media"] = reav.drop(index="Sem divisão (Brasil)")[[c for c in reav.columns if c.startswith("R2")]].rank(ascending=False).mean(axis=1)
reav.sort_values("media", ascending=False).round(3)

## 15. Síntese: defasagem e nível de evidência por região
A cor indica a defasagem (meses) dos padrões **confirmados** (evidência forte ou moderada). Regiões **hachuradas** tinham um padrão candidato que **não se confirmou** em 2021–2025; as cinzas não têm padrão consistente.

In [ ]:
def mapa_sintese(res_bioma: pd.DataFrame, res_uf: pd.DataFrame, subtitulo: str = "") -> None:
    fig, axes = plt.subplots(1, 2, figsize=(18, 9), layout="constrained")
    norma = plt.Normalize(0, 9)
    for ax, (nome, res) in zip(axes, [(P_LAG["divisao_principal"], res_bioma),
                                       (P_LAG["divisao_sensibilidade"], res_uf)]):
        rot = divisoes[nome]
        g = micro.assign(regiao=micro["CD_MICRO"].map(rot)).dissolve(by="regiao").reset_index()
        g = g.merge(res.rename(columns={"região": "regiao"}), on="regiao", how="left")
        micro.plot(ax=ax, color="#f2f2f2", linewidth=0)
        g[g["nível"] == "sem padrão consistente"].plot(ax=ax, color="#d9d9d9", edgecolor="white", linewidth=0.4)
        conf = g[g["nível"].isin(["forte", "moderada"])]
        conf.plot(ax=ax, column="defasagem (meses)", cmap="viridis", norm=norma, edgecolor="black", linewidth=0.5)
        nc = g[g["nível"] == "não confirmada"]
        if len(nc):
            nc.plot(ax=ax, facecolor="white", hatch="///", edgecolor="grey", linewidth=0.4)
        for _, r in conf.iterrows():
            pt = r.geometry.representative_point()
            marca = "●" if r["nível"] == "forte" else "○"
            ax.annotate(f"{marca} {r['regiao']}\n{int(r['defasagem (meses)'])} m · ρ={r['ρ exploração']:+.2f}",
                        (pt.x, pt.y), ha="center", fontsize=8,
                        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.75, lw=0))
        ax.set_title(f"{nome} {subtitulo}\n● evidência forte · ○ moderada · hachurado = não confirmada · cinza = sem padrão")
        ax.axis("off")
    fig.colorbar(plt.cm.ScalarMappable(cmap="viridis", norm=norma), ax=list(axes), shrink=0.5,
                 label="defasagem (meses antes do mês da chuva)")
    plt.show()


mapa_sintese(resultado_bioma, resultado_uf, "· níveis do D-03 (FDR)")

## 16. Conclusões
**Defasagem e nível de evidência por região** (ONI; ρ na exploração, safras até 2020, e na verificação, 2021–2025):

| Região | Defasagem | ρ exploração | ρ 2021–2025 | Nível | Soja (Mt/ano) |
|---|---|---|---|---|---|
| **Cerrado** (bioma) | **7 meses** (janela 3) | −0,29 | −0,28 | moderada | 54,9 |
| Amazônia (bioma) | 2 meses (janela 6) | −0,34 | −0,38 | **forte** | 12,5 |
| Pampa (bioma) | 0 meses | +0,22 | +0,41 | moderada | 7,7 |
| Caatinga (bioma) | 3 meses (janela 6) | −0,22 | +0,06 | não confirmada | 0,04 |
| Mata Atlântica (bioma) | — | — | — | sem padrão consistente | 18,4 |
| **MT** (UF) | 5 meses (janela 6) | −0,41 | −0,16 | **forte** | 29,4 |
| GO (UF) | 8 meses | −0,28 | −0,13 | moderada | 10,9 |
| MG (UF) | 8 meses (janela 3) | −0,17 | −0,14 | moderada | 5,1 |
| MS (UF) | 9 meses | −0,19 | −0,04 | não confirmada | 8,5 |
| RS, PR, SC (UF) | — | — | — | sem padrão consistente | 23,5 |

> **Atualização (Parte 3, D-04 aprovado):** o critério de evidência "forte" passou a ser o teste por séries substitutas. **Níveis vigentes: Cerrado forte, GO forte, MT forte, Amazônia moderada**; os demais não mudam. Ver a seção 17 e o mapa-síntese revisado.

**O que os dados mostram:**
1. **Gangorra Norte–Sul:** em anos de El Niño, a chuva da safra tende a ser **menor** no Norte e no Centro-Oeste e **maior** no Sul (Pampa). É o padrão descrito na literatura sobre o ENSO no Brasil.
2. **Resposta imediata no Sul e na Amazônia, atrasada no Cerrado:**
   - no Pampa e no PA, a relação aparece **sem defasagem**;
   - no Cerrado e nas UFs dele (MT, GO, MG), a relação mais forte aparece com **5 a 8 meses** de defasagem, e quase não existe sem defasagem (ρ = −0,01 na Parte 1);
   - ou seja, a chuva da safra no Cerrado se relaciona com o estado do Pacífico vários meses antes (por exemplo, a chuva de janeiro com o ONI de abril a junho do ano anterior).
3. **O Cerrado é o resultado mais relevante:** concentra a maior parte da soja da base e é onde a defasagem faz mais diferença. Na verificação de 2021–2025, o ρ com 7 meses de defasagem se repetiu (−0,28), enquanto sem defasagem ficou em −0,06.

**Grau de confiança:**
- as correlações são **moderadas** (|ρ| entre 0,15 e 0,4): o ENSO explica uma parte da variação da chuva, não a maior parte;
- só a Amazônia (bioma) e o MT passam também no critério formal (FDR); os demais são evidência **moderada**: um padrão consistente que se repetiu em outro período;
- a verificação usou só 5 safras, e quase todos os intervalos de confiança de 2021–2025 cruzam o zero (a exceção é o PA). Ela mostra que o **sinal se manteve**, não que é significativo nesse período curto;
- no MT, a relação se repetiu mais fraca (−0,41 → −0,16), e a correlação sem defasagem mudou de sinal em 2021–2025 (+0,32), o que pede cautela;
- no **Sul**, o sinal sem defasagem é claro no Pampa, mas **instável** nas UFs (RS, PR e SC mudam de comportamento entre as safras pares e as ímpares).

**Hipótese a investigar:** a defasagem no Cerrado pode refletir o efeito do ENSO no **início da estação chuvosa** (setembro a novembro), já que o evento começa a se formar no outono e no inverno. Repetir a análise separando o início, o meio e o fim da safra ajudaria a testar isso.

**Limitações:**
- a análise usa a chuva, não o rendimento;
- só cerca de 20 safras;
- a resposta de cada microrregião isolada é dominada por ruído, e o sinal só fica confiável quando agregado por região;
- os critérios de consistência (platô, SOI e estabilidade) foram definidos durante a exploração, embora antes da verificação.

**Próximos passos:**
- repetir a análise com a anomalia de rendimento (depois do `desvio_clima` por microrregião);
- separar a safra em subperíodos;
- incorporar ao pipeline o ONI defasado por bioma para as regiões com evidência forte ou moderada.

---
# Parte 3 · Testes de robustez e aprofundamento

Ordem: **17.** séries substitutas · **18.** subperíodos da safra · **19.** fases do ENSO · (depois) sensibilidade às escolhas e outras variáveis. Decisões em `docs/decisoes.md` (D-04 em diante).

## 17. Teste de significância por séries substitutas
**Por que:** o FDR (D-02) trata as 195 combinações de defasagem × janela como testes independentes. Elas não são (defasagens vizinhas e janelas sobrepostas medem quase o mesmo), e o ONI é muito autocorrelacionado, então o FDR fica conservador demais.

**Como:**
- **Hipótese nula:** a chuva da região não tem relação com o ENSO.
- **Estatística:** em cada região, o **maior |ρ| entre todas as combinações** de defasagem × janela, a mesma busca feita na Parte 2. Assim o teste já desconta o fato de termos escolhido a melhor combinação.
- **Distribuição sob a nula:** a mesma estatística calculada com séries ENSO **substitutas**, que mantêm as propriedades do ONI real mas não têm relação com a chuva:
  - **randomização de fase** (Theiler et al., 1992), 2.000 séries: preserva o espectro e a autocorrelação do ONI, mas **não** o ciclo sazonal da variância do ENSO;
  - **deslocamento circular por anos inteiros**, 71 séries (deslocamentos de 3 a 73 anos na série 1950–2025): preserva **tudo**, inclusive o ciclo sazonal, com menos resolução.
- **p-valor:** p = (1 + nº de substitutas com máx|ρ| ≥ observado) / (N + 1).
- **Período:** só as safras da exploração (até 2020), como no D-02.

**Critério (fixado antes da execução):**
- **passa:** p < 0,05 nas **duas** famílias;
- **indício:** p < 0,05 em apenas uma delas;
- **não passa:** nas demais.

Também é calculado um **p global por divisão** (o maior |ρ| entre todas as regiões), que responde: "existe relação em alguma região?".

In [ ]:
from scipy.stats import rankdata

P_SUB = dict(n_fase=2000, desloc_min_anos=3, seed=42, ano_ini=1950, ano_fim=2025)
COMBOS = [(L, w) for w in P_LAG["janelas"] for L in P_LAG["lags"]]


def matriz_indice(serie: pd.Series, meses: pd.PeriodIndex) -> np.ndarray:
    # meses × combinações (L, w): média do índice nos w meses que terminam L meses antes
    return np.column_stack([serie.rolling(w, min_periods=w).mean().shift(L).reindex(meses).to_numpy()
                            for L, w in COMBOS])


def rho_todas(X: np.ndarray, Y: pd.DataFrame) -> np.ndarray:
    # Spearman (correlação dos postos) de cada combinação com cada região: regiões × combinações
    out = np.full((Y.shape[1], X.shape[1]), np.nan)
    for j, reg in enumerate(Y.columns):
        y = Y[reg].to_numpy()
        ok = ~np.isnan(y) & ~np.isnan(X).any(axis=1)
        if ok.sum() < 24:
            continue
        ry = rankdata(y[ok]); ry = (ry - ry.mean()) / ry.std()
        rx = rankdata(X[ok], axis=0); rx = (rx - rx.mean(0)) / rx.std(0)
        out[j] = (rx * ry[:, None]).mean(0)
    return out


def substitutas_fase(x: np.ndarray, n: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    media = x.mean()
    F = np.fft.rfft(x - media)
    amp = np.abs(F)
    out = np.empty((n, len(x)))
    for i in range(n):
        fase = rng.uniform(0, 2 * np.pi, len(F))
        fase[0] = 0
        if len(x) % 2 == 0:
            fase[-1] = 0
        out[i] = np.fft.irfft(amp * np.exp(1j * fase), len(x)) + media
    return out


# ONI em anos completos (o deslocamento por 12k meses mantém o calendário alinhado)
oni_ref = enso["oni"][(enso.index.year >= P_SUB["ano_ini"]) & (enso.index.year <= P_SUB["ano_fim"])]
assert not oni_ref.isna().any() and len(oni_ref) % 12 == 0
x_oni = oni_ref.to_numpy()
subs = {
    "fase": substitutas_fase(x_oni, P_SUB["n_fase"], P_SUB["seed"]),
    "deslocamento": np.array([np.roll(x_oni, 12 * k)
                              for k in range(P_SUB["desloc_min_anos"], len(x_oni) // 12 - P_SUB["desloc_min_anos"] + 1)]),
}

# Verificação: as substitutas preservam a autocorrelação e (só o deslocamento) o ciclo sazonal da variância?
def _resumo(arr):
    ac = lambda s, k: np.corrcoef(s[:-k], s[k:])[0, 1]
    var_mes = np.array([[s[m::12].var() for m in range(12)] for s in arr]).mean(0)
    return dict(autocorr_1m=np.mean([ac(s, 1) for s in arr]), autocorr_6m=np.mean([ac(s, 6) for s in arr]),
                razao_var_dezjan_maijun=var_mes[[11, 0]].mean() / var_mes[[4, 5]].mean())
pd.DataFrame({"ONI real": _resumo(x_oni[None]), **{k: _resumo(v) for k, v in subs.items()}}).T.round(2)

In [ ]:
def teste_substitutas(nome: str) -> tuple[pd.DataFrame, dict]:
    Y = series_regionais(divisoes[nome])[explora]
    rho_obs = rho_todas(matriz_indice(oni_ref, Y.index), Y)
    max_obs = np.nanmax(np.abs(rho_obs), axis=1)
    melhor = np.nanargmax(np.abs(rho_obs), axis=1)
    nulos = {}
    for fam, arr in subs.items():
        nulos[fam] = np.array([np.nanmax(np.abs(rho_todas(matriz_indice(pd.Series(s, index=oni_ref.index), Y.index), Y)), axis=1)
                               for s in arr])            # substitutas × regiões
    rot = divisoes[nome].reindex(anom.columns).dropna()
    prod = producao.reindex(rot.index).groupby(rot).sum() / 1e6
    linhas = []
    for j, reg in enumerate(Y.columns):
        L, w = COMBOS[melhor[j]]
        p = {fam: (1 + (nulos[fam][:, j] >= max_obs[j]).sum()) / (len(nulos[fam]) + 1) for fam in nulos}
        n_sig = sum(v < 0.05 for v in p.values())
        linhas.append({"região": reg, "soja (Mt)": round(prod.get(reg, np.nan), 1),
                       "máx|ρ| observado": max_obs[j], "em (defasagem, janela)": f"{L} m, j{w}",
                       "ρ": rho_obs[j, melhor[j]],
                       "limiar 95% (fase)": np.quantile(nulos["fase"][:, j], 0.95),
                       "p (fase)": p["fase"], "p (deslocamento)": p["deslocamento"],
                       "resultado": "passa" if n_sig == 2 else ("indício" if n_sig == 1 else "não passa")})
    tab = pd.DataFrame(linhas)
    # correção pelo número de regiões testadas (BH), acrescentada depois da primeira execução: ver o texto abaixo
    for fam in ("fase", "deslocamento"):
        tab[f"p BH entre regiões ({fam})"] = multipletests(tab[f"p ({fam})"], method="fdr_bh")[1]
    tab = tab.sort_values("p (fase)")
    # p global da divisão: maior |ρ| entre todas as regiões
    glob = {fam: (1 + (nulos[fam].max(axis=1) >= max_obs.max()).sum()) / (len(nulos[fam]) + 1) for fam in nulos}
    return tab, dict(nulos=nulos, max_obs=max_obs, regioes=list(Y.columns), p_global=glob)


subst = {}
for nome in (P_LAG["divisao_principal"], P_LAG["divisao_sensibilidade"]):
    tab, extra = teste_substitutas(nome)
    subst[nome] = (tab, extra)
    print(f"{nome}: p global (fase) = {extra['p_global']['fase']:.4f} | p global (deslocamento) = {extra['p_global']['deslocamento']:.4f}")
subst[P_LAG["divisao_principal"]][0].round(3)

In [ ]:
# UFs: principais produtoras (soja >= 1 Mt/ano) e todas as que passam ou têm indício
tab_uf = subst[P_LAG["divisao_sensibilidade"]][0]
tab_uf[(tab_uf["soja (Mt)"] >= 1) | (tab_uf["resultado"] != "não passa")].round(3)

In [ ]:
# Comparação com os níveis de evidência do D-03 (revisão aprovada: D-04)
def niveis_revisados(nome: str, resultado_d03: pd.DataFrame) -> pd.DataFrame:
    t = subst[nome][0].set_index("região")
    r = resultado_d03.set_index("região")[["defasagem (meses)", "ρ exploração", "ρ 2021–2025", "nível"]].rename(
        columns={"nível": "nível D-03 (FDR)"})
    r = r.join(t[["p (fase)", "p (deslocamento)", "p BH entre regiões (fase)", "resultado"]].rename(
        columns={"resultado": "teste substitutas"}))
    def novo(l):
        if l["nível D-03 (FDR)"] in ("sem padrão consistente",):
            return l["nível D-03 (FDR)"]
        if l["nível D-03 (FDR)"] == "não confirmada":
            return "não confirmada"
        return "forte" if l["teste substitutas"] == "passa" else "moderada"
    r["nível revisado (substitutas)"] = r.apply(novo, axis=1)
    return r


niveis_revisados(P_LAG["divisao_principal"], resultado_bioma).round(3)

In [ ]:
# Mapa-síntese com os níveis VIGENTES (revisados pelo D-04, aprovado)
def aplicar_revisao(nome: str, resultado: pd.DataFrame) -> pd.DataFrame:
    rev = niveis_revisados(nome, resultado)["nível revisado (substitutas)"]
    return resultado.assign(**{"nível": resultado["região"].map(rev)})


resultado_bioma_rev = aplicar_revisao(P_LAG["divisao_principal"], resultado_bioma)
resultado_uf_rev = aplicar_revisao(P_LAG["divisao_sensibilidade"], resultado_uf)
mapa_sintese(resultado_bioma_rev, resultado_uf_rev, "· níveis vigentes (D-04)")
resultado_uf_rev[resultado_uf_rev["nível"] != "sem padrão consistente"][["região", "soja (Mt)", "defasagem (meses)", "nível"]]

**Leitura e cautelas** (acrescentadas depois da primeira execução, por transparência):
- **Cerrado, MT e GO passam** nas duas famílias de substitutas, e esse resultado já desconta a busca entre as 39 combinações. O Cerrado também se repetiu no período independente (2021–2025; seção 12).
- **Correção entre regiões:** o critério fixado era por região. Corrigindo também para o número de regiões testadas (BH), o p do Cerrado sobe para 0,07–0,08 (5 biomas), e o do MT e do GO para 0,13–0,29 (21 UFs). A evidência do Cerrado se apoia, por isso, em **três pilares juntos**:
  1. p ≈ 0,015 na exploração;
  2. repetição em 2021–2025;
  3. coerência com as UFs do Cerrado (MT e GO passam individualmente).
- **A Amazônia não passa** (p ≈ 0,09), embora tenha passado no FDR do índice combinado (D-02). A série regional dela tem menos meses válidos e é mais ruidosa, e o teste por substitutas, que respeita a autocorrelação, exige um |ρ| maior (limiar de 0,37) do que o FDR exigia.
- **p global:** o maior |ρ| entre todas as regiões (p ≈ 0,10) é dominado pelas regiões mais ruidosas, que têm limiares mais altos. Por isso, os resultados por região são os principais.

In [ ]:
# Distribuição nula do máx|ρ| (randomização de fase) por bioma, com o valor observado
tab_b, ext_b = subst[P_LAG["divisao_principal"]]
fig, axes = plt.subplots(1, len(ext_b["regioes"]), figsize=(4 * len(ext_b["regioes"]), 3.2), sharey=True, layout="constrained")
for ax, (j, reg) in zip(np.ravel([axes]), enumerate(ext_b["regioes"])):
    nulo = ext_b["nulos"]["fase"][:, j]
    ax.hist(nulo, bins=40, color="#bbbbbb")
    ax.axvline(np.quantile(nulo, 0.95), color="grey", ls="--", label="95% da nula")
    ax.axvline(ext_b["max_obs"][j], color="#b2182b", lw=2, label="observado")
    lin = tab_b.set_index("região").loc[reg]
    ax.set_title(f"{reg}\np fase={lin['p (fase)']:.3f} · p desloc.={lin['p (deslocamento)']:.3f}", fontsize=10)
    ax.set_xlabel("máx|ρ| entre as 39 combinações")
np.ravel([axes])[0].legend(fontsize=8)
fig.suptitle("Teste por séries substitutas (exploração, safras até 2020)")
plt.show()

## 18. Subperíodos da safra
**Hipótese (D-03):** a relação defasada no Cerrado vem principalmente do **início da estação chuvosa**. A literatura indica que o efeito do ENSO no Centro-Leste do Brasil pode mudar ao longo da estação, e a média de setembro a abril pode estar misturando efeitos diferentes.

**Como:**
- **(a)** a varredura de defasagens (0 a 12 meses × janelas de 1, 3 e 6) é repetida em cada subperíodo: **início** (set–nov), **meio** (dez–fev) e **fim** (mar–abr). A significância usa o **teste por substitutas da seção 17** (máx\|ρ\| entre as 39 combinações, duas famílias).
- **(b)** para o Cerrado, o MT e o GO, uma matriz **mês da chuva × mês do ONI** (janela de 1 mês). Ela mostra se a relação está presa a uma **época do ENSO** (por exemplo, o ONI de outono ou inverno), caso em que a "defasagem" seria uma consequência dessa época.

**Critério (fixado antes da execução):** a hipótese é **apoiada** se, no Cerrado, o máx\|ρ\| do **início** for maior que o do meio e o do fim **e** o início **passar** no teste por substitutas (p < 0,05 nas duas famílias).

**Limitação:** cada subperíodo tem cerca de 48 meses na exploração (32 no fim), contra 124 da safra inteira. Os resultados são mais ruidosos, e a verificação em 2021–2025 (só cerca de 15 meses por subperíodo) não é feita.

In [ ]:
SUBPERIODOS = {"início (set–nov)": [9, 10, 11], "meio (dez–fev)": [12, 1, 2], "fim (mar–abr)": [3, 4]}


def teste_subperiodo(nome: str, meses_cal: list[int]) -> pd.DataFrame:
    mascara = explora & np.isin(anom.index.month, meses_cal)
    Y = series_regionais(divisoes[nome])[mascara]
    rho_obs = rho_todas(matriz_indice(oni_ref, Y.index), Y)
    max_obs = np.nanmax(np.abs(rho_obs), axis=1)
    melhor = np.nanargmax(np.abs(rho_obs), axis=1)
    nulos = {fam: np.array([np.nanmax(np.abs(rho_todas(matriz_indice(pd.Series(s, index=oni_ref.index), Y.index), Y)), axis=1)
                            for s in arr]) for fam, arr in subs.items()}
    linhas = []
    for j, reg in enumerate(Y.columns):
        L, w = COMBOS[melhor[j]]
        p = {fam: (1 + (nulos[fam][:, j] >= max_obs[j]).sum()) / (len(nulos[fam]) + 1) for fam in nulos}
        n_sig = sum(v < 0.05 for v in p.values())
        linhas.append({"região": reg, "meses": int(Y[reg].notna().sum()), "máx|ρ|": max_obs[j], "ρ": rho_obs[j, melhor[j]],
                       "defasagem": L, "janela": w, "p (fase)": p["fase"], "p (deslocamento)": p["deslocamento"],
                       "resultado": "passa" if n_sig == 2 else ("indício" if n_sig == 1 else "não passa")})
    return pd.DataFrame(linhas), rho_obs, list(Y.columns)


res_sub, rho_sub = {}, {}
for sp, meses_cal in SUBPERIODOS.items():
    t, r, regs = teste_subperiodo(P_LAG["divisao_principal"], meses_cal)
    res_sub[sp] = t.assign(subperiodo=sp)
    rho_sub[sp] = (r, regs)
tab_sub = pd.concat(res_sub.values())
tab_sub.pivot(index="região", columns="subperiodo", values=["ρ", "defasagem", "p (fase)", "resultado"])[
    [(m, sp) for m in ["ρ", "defasagem", "p (fase)", "resultado"] for sp in SUBPERIODOS]].round(3)

In [ ]:
# Avaliação da hipótese, com o critério fixado acima
cer = tab_sub[tab_sub["região"] == "Cerrado"].set_index("subperiodo")
inicio = cer.loc["início (set–nov)"]
apoiada = bool(inicio["máx|ρ|"] > cer.drop(index="início (set–nov)")["máx|ρ|"].max() and inicio["resultado"] == "passa")
print("Cerrado, máx|ρ| por subperíodo:", cer["máx|ρ|"].round(3).to_dict())
print("Resultado do teste no início:", inicio["resultado"], f"(p fase = {inicio['p (fase)']:.3f}, p deslocamento = {inicio['p (deslocamento)']:.3f})")
print("HIPÓTESE APOIADA" if apoiada else "HIPÓTESE NÃO APOIADA pelo critério fixado")

In [ ]:
# (a) mapas de calor por subperíodo: região × defasagem, janela de 3 meses
w_fig = 3
fig, axes = plt.subplots(1, len(SUBPERIODOS), figsize=(6 * len(SUBPERIODOS), 3.6), sharey=True, layout="constrained")
for ax, (sp, (r, regs)) in zip(axes, rho_sub.items()):
    cols = [k for k, (L, w) in enumerate(COMBOS) if w == w_fig]
    m = pd.DataFrame(r[:, cols], index=regs, columns=[COMBOS[k][0] for k in cols])
    ordem = ["Amazônia", "Caatinga", "Cerrado", "Mata Atlântica", "Pampa"]
    m = m.reindex([o for o in ordem if o in m.index])
    im = ax.imshow(m.values, cmap="BrBG", vmin=-0.45, vmax=0.45, aspect="auto")
    ax.set_xticks(range(m.shape[1]), m.columns)
    ax.set_yticks(range(m.shape[0]), m.index)
    ax.set_xlabel("defasagem (meses)")
    res = res_sub[sp].set_index("região")
    ax.set_title(f"{sp} · Cerrado: {res.loc['Cerrado', 'resultado']}")
fig.colorbar(im, ax=list(axes), shrink=0.8, label=f"ρ (ONI defasado, janela {w_fig} m × anomalia de chuva)")
fig.suptitle("Defasagens por subperíodo da safra (exploração, safras até 2020)")
plt.show()

In [ ]:
# (b) mês da chuva × mês do ONI (janela de 1 mês): a relação está presa a uma época do ENSO?
MESES_CHUVA = [9, 10, 11, 12, 1, 2, 3, 4]
NOMES = {1: "jan", 2: "fev", 3: "mar", 4: "abr", 5: "mai", 6: "jun", 7: "jul", 8: "ago", 9: "set", 10: "out", 11: "nov", 12: "dez"}


def matriz_mes_a_mes(serie_regional: pd.Series) -> pd.DataFrame:
    # linhas: mês da chuva; colunas: defasagem 0-12 -> convertida no mês do ONI
    out = {}
    for mc in MESES_CHUVA:
        mascara = explora & (anom.index.month == mc)
        y = serie_regional[mascara]
        out[mc] = {L: stats.spearmanr(oni_ref.shift(L).reindex(y.index), y, nan_policy="omit").statistic for L in range(13)}
    return pd.DataFrame(out).T   # mês da chuva × defasagem


regioes_b = {"Cerrado (bioma)": series_regionais(divisoes[P_LAG["divisao_principal"]])["Cerrado"],
             "MT": series_regionais(divisoes[P_LAG["divisao_sensibilidade"]])["MT"],
             "GO": series_regionais(divisoes[P_LAG["divisao_sensibilidade"]])["GO"]}
fig, axes = plt.subplots(1, 3, figsize=(19, 4.6), layout="constrained")
for ax, (nome_r, serie) in zip(axes, regioes_b.items()):
    m = matriz_mes_a_mes(serie)
    im = ax.imshow(m.values, cmap="BrBG", vmin=-0.6, vmax=0.6, aspect="auto")
    ax.set_yticks(range(len(m.index)), [NOMES[x] for x in m.index])
    ax.set_xticks(range(13), range(13))
    ax.set_xlabel("defasagem (meses)")
    ax.set_ylabel("mês da chuva")
    # anotar o mês do ONI correspondente nas células com |ρ| alto
    for i, mc in enumerate(m.index):
        for L in range(13):
            if abs(m.iloc[i, L]) >= 0.4:
                ax.text(L, i, NOMES[(mc - L - 1) % 12 + 1], ha="center", va="center", fontsize=7)
    ax.set_title(f"{nome_r} · células com |ρ| ≥ 0,4 mostram o mês do ONI")
fig.colorbar(im, ax=list(axes), shrink=0.8, label="ρ (ONI do mês t−L × chuva do mês t)")
fig.suptitle("Mês da chuva × defasagem do ONI (exploração; cerca de 16 safras por célula)")
plt.show()

**Checagem acrescentada depois de ver a matriz (b):** o ENSO tende a alternar de fase de um ano para o outro (depois de um El Niño é comum vir um La Niña). A relação com o ONI de 7 meses antes poderia, então, ser só um **retrato indireto do estado atual do ENSO**, e não um efeito defasado. Para testar, calcula-se a **correlação parcial** entre o ONI defasado e a chuva **controlando o ONI do mesmo mês**.

In [ ]:
def parcial_spearman(y: pd.Series, x: pd.Series, z: pd.Series) -> float:
    d = pd.concat([y, x, z], axis=1).dropna()
    r = d.apply(rankdata)
    resid = lambda a, b: a - np.polyval(np.polyfit(b, a, 1), b)
    return float(np.corrcoef(resid(r.iloc[:, 1], r.iloc[:, 2]), resid(r.iloc[:, 0], r.iloc[:, 2]))[0, 1])


casos = [("Cerrado (bioma)", P_LAG["divisao_principal"], "Cerrado", 7, 3),
         ("MT", P_LAG["divisao_sensibilidade"], "MT", 7, 3),
         ("GO", P_LAG["divisao_sensibilidade"], "GO", 8, 1)]
lin = []
for rotulo, div, reg, L, w in casos:
    y = series_regionais(divisoes[div])[reg][explora]
    x_def, x_atual = enso_janela("oni", L, w).reindex(y.index), enso_janela("oni", 0, 1).reindex(y.index)
    lin.append({"região": rotulo, "defasagem": f"{L} m, j{w}",
                "ρ ONI defasado": stats.spearmanr(x_def, y, nan_policy="omit").statistic,
                "ρ ONI atual": stats.spearmanr(x_atual, y, nan_policy="omit").statistic,
                "ρ entre os dois índices": stats.spearmanr(x_def, x_atual, nan_policy="omit").statistic,
                "ρ parcial (defasado | atual)": parcial_spearman(y, x_def, x_atual)})
pd.DataFrame(lin).round(3)

### Leitura da seção 18
- **Hipótese não apoiada.** No Cerrado, o maior |ρ| aparece no **fim** da safra (mar–abr: −0,41, com 9 meses), e não no início (−0,28). Nenhum subperíodo passa no teste por substitutas: com cerca de 32 a 48 meses, a variação ao acaso é grande.
- **O efeito muda de sinal ao longo da safra** (matriz b):
  - em **setembro**, a chuva no Cerrado, no MT e no GO **aumenta** com o ONI do mesmo período (ρ > +0,4);
  - de **dezembro a abril**, ela **diminui** com o ONI de muitos meses antes (o ONI do verão e do outono anteriores).

  Por isso a correlação sem defasagem na safra inteira é quase zero: **efeitos de sinais opostos se cancelam** na média de setembro a abril.
- **A relação defasada não é um retrato do estado atual do ENSO.** Controlando o ONI do mesmo mês, a correlação parcial praticamente não muda (Cerrado −0,30; MT −0,39; GO −0,29), e o ONI defasado é **positivamente** correlacionado com o atual (+0,21), o oposto do que a alternância de fases produziria.
- **Implicação para as próximas etapas:** agregar a chuva da safra inteira esconde a mudança de sinal. Para o Cerrado, vale tratar o **início** (setembro) e o **resto da safra** separadamente, o que conversa com as fases fenológicas que já existem no pipeline.
- **Cautela:** cada célula da matriz mês × mês tem só cerca de 16 safras, então ela serve para mostrar o **padrão** e não para afirmar significância célula a célula.

## 19. Comparações por fase do ENSO (El Niño × La Niña × neutro)
**Por que:** a correlação mede uma relação linear e supõe que o efeito do La Niña é o espelho do El Niño. Comparar as safras **por fase** é mais fácil de comunicar ("em anos de La Niña, X% menos chuva"), permite ver **assimetria** e mostra o efeito da **intensidade** do evento.

**Como:**
- **Meses de El Niño / La Niña** (definição da NOAA): ONI ≥ +0,5 / ≤ −0,5 por pelo menos 5 meses seguidos.
- **Fase da safra:** El Niño ou La Niña se **pelo menos 4 dos 8 meses** (set–abr) estão nessa fase; caso contrário, neutra. **Intensidade** pelo pico do \|ONI\| na safra: fraco (0,5–0,9), moderado (1,0–1,4) e forte (≥ 1,5).
- **Resposta:** anomalia da **chuva total da safra em % da média** de cada microrregião (exige pelo menos 7 dos 8 meses com dado e 10 safras para a média), depois a média das microrregiões da região.
- **Período:** todas as safras (2006–2025). É uma comparação **descritiva**: precisa de todos os eventos disponíveis.
- **Incerteza:** IC de 95% por bootstrap das safras dentro de cada fase; **teste de permutação** dos rótulos de fase (10.000 permutações).
- **Referência:** a **média de todas as safras** (anomalia zero), e a diferença **El Niño − La Niña**. Só 4 safras são neutras, pouco para servirem de referência (ajuste decidido antes de calcular os resultados).
- **Robusto:** o IC de 95% não cruza o zero **e** o p da permutação é menor que 0,05.

In [ ]:
def fase_mensal(oni: pd.Series, limiar: float = 0.5, n_min: int = 5) -> pd.Series:
    out = pd.Series("neutro", index=oni.index)
    for nome, cond in (("El Niño", oni >= limiar), ("La Niña", oni <= -limiar)):
        grupo = (cond != cond.shift()).cumsum()
        out[cond & (cond.groupby(grupo).transform("size") >= n_min)] = nome
    return out


oni_m = enso["oni"].dropna()
fm = fase_mensal(oni_m)
safra_m = pd.Series(np.where(oni_m.index.month >= 9, oni_m.index.year + 1, oni_m.index.year), index=oni_m.index)
na_janela = np.isin(oni_m.index.month, P["meses_safra"])
lin = []
for s_, idx in fm[na_janela].groupby(safra_m[na_janela]).groups.items():
    if s_ < 2006 or s_ > 2025:
        continue
    cont = fm[idx].value_counts()
    pico = oni_m[idx][oni_m[idx].abs().idxmax()]
    fase = "El Niño" if cont.get("El Niño", 0) >= 4 else ("La Niña" if cont.get("La Niña", 0) >= 4 else "neutro")
    inten = "" if fase == "neutro" else ("forte" if abs(pico) >= 1.5 else "moderado" if abs(pico) >= 1.0 else "fraco")
    lin.append(dict(safra=int(s_), rotulo=f"{s_ - 1}/{str(s_)[2:]}", fase=fase, intensidade=inten, oni_pico=pico))
fases = pd.DataFrame(lin).set_index("safra")
print(fases["fase"].value_counts().to_dict())
fases

In [ ]:
def anomalia_pct(meses_cal: list[int], min_meses: int) -> pd.DataFrame:
    # anomalia % da chuva total no subperíodo, por microrregião e safra
    cm = chuva_micro[chuva_micro["mes"].isin(meses_cal)].copy()
    cm["safra"] = np.where(cm["mes"] >= 9, cm["mes_ref"].dt.year + 1, cm["mes_ref"].dt.year)
    tot = cm.groupby(["CD_MICRO", "safra"]).agg(total=("chuva", "sum"), n=("chuva", "size")).reset_index()
    tot = tot[tot["n"] >= min_meses]
    tot = tot[tot.groupby("CD_MICRO")["safra"].transform("size") >= 10]
    tot["anom_pct"] = 100 * (tot["total"] / tot.groupby("CD_MICRO")["total"].transform("mean") - 1)
    return tot.pivot(index="safra", columns="CD_MICRO", values="anom_pct")


def regional_pct(anom_micro: pd.DataFrame, nome: str, micros_validas=None) -> pd.DataFrame:
    validas = anom.columns if micros_validas is None else micros_validas
    rot = divisoes[nome].reindex([c for c in anom_micro.columns if c in validas]).dropna()
    return pd.DataFrame({reg: anom_micro[mics].mean(axis=1) for reg, mics in rot.groupby(rot).groups.items()})


ANOM_SAFRA = anomalia_pct(P["meses_safra"], min_meses=7)
reg_pct = regional_pct(ANOM_SAFRA, P_LAG["divisao_principal"]).reindex(fases.index)
print(f"Microrregiões com anomalia % da safra: {ANOM_SAFRA.shape[1]} | safras: {ANOM_SAFRA.index.min()}–{ANOM_SAFRA.index.max()}")


def comparar_fases(serie: pd.Series, n_boot: int = 5000, n_perm: int = 10000, seed: int = 42) -> dict:
    rng = np.random.default_rng(seed)
    d = pd.DataFrame({"y": serie, "fase": fases["fase"]}).dropna()
    out = {}
    for f in ("El Niño", "La Niña", "neutro"):
        y = d.loc[d["fase"] == f, "y"].to_numpy()
        boot = [rng.choice(y, len(y)).mean() for _ in range(n_boot)]
        perm = np.array([rng.permutation(d["y"].to_numpy())[: len(y)].mean() for _ in range(n_perm)])
        p = (1 + (np.abs(perm - d["y"].mean()) >= abs(y.mean() - d["y"].mean())).sum()) / (n_perm + 1)
        out[f] = dict(media=y.mean(), ic_lo=np.percentile(boot, 2.5), ic_hi=np.percentile(boot, 97.5), p=p, n=len(y))
    en, ln = d.loc[d["fase"] == "El Niño", "y"].to_numpy(), d.loc[d["fase"] == "La Niña", "y"].to_numpy()
    obs = en.mean() - ln.mean()
    todos = np.concatenate([en, ln])
    perm = np.array([(lambda z: z[: len(en)].mean() - z[len(en):].mean())(rng.permutation(todos)) for _ in range(n_perm)])
    out["dif"] = dict(media=obs, p=(1 + (np.abs(perm) >= abs(obs)).sum()) / (n_perm + 1))
    return out


linhas = []
for reg in reg_pct.columns:
    c = comparar_fases(reg_pct[reg])
    lin_ = {"região": reg}
    for f in ("El Niño", "La Niña", "neutro"):
        robusto = (c[f]["ic_lo"] > 0 or c[f]["ic_hi"] < 0) and c[f]["p"] < 0.05
        lin_[f"{f} (%)"] = f"{c[f]['media']:+.1f} [{c[f]['ic_lo']:+.1f}; {c[f]['ic_hi']:+.1f}]" + (" *" if robusto else "")
        lin_[f"p {f}"] = c[f]["p"]
    lin_["El Niño − La Niña (p.p.)"] = c["dif"]["media"]
    lin_["p diferença"] = c["dif"]["p"]
    lin_["assimetria |EN|/|LN|"] = abs(c["El Niño"]["media"]) / max(abs(c["La Niña"]["media"]), 1e-9)
    linhas.append(lin_)
comp_bioma = pd.DataFrame(linhas)
print("* = robusto (IC 95% não cruza zero e p < 0,05). n safras: El Niño 7, La Niña 9, neutro 4.")
comp_bioma.round(3)

In [ ]:
# Gráfico: anomalia média da chuva da safra por fase e bioma (IC 95%)
cores = {"El Niño": "#c0392b", "neutro": "#999999", "La Niña": "#2471a3"}
fig, ax = plt.subplots(figsize=(11, 4.5), layout="constrained")
regs = list(reg_pct.columns)
for k, f in enumerate(("El Niño", "neutro", "La Niña")):
    meds, los, his = [], [], []
    for reg in regs:
        c = comparar_fases(reg_pct[reg], n_perm=200)[f]
        meds.append(c["media"]); los.append(c["media"] - c["ic_lo"]); his.append(c["ic_hi"] - c["media"])
    x = np.arange(len(regs)) + (k - 1) * 0.27
    ax.bar(x, meds, width=0.25, color=cores[f], label=f)
    ax.errorbar(x, meds, yerr=[los, his], fmt="none", ecolor="black", capsize=3, lw=1)
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(range(len(regs)), regs)
ax.set_ylabel("anomalia da chuva da safra (% da média)")
ax.set_title("Chuva da safra (set–abr) por fase do ENSO, safras 2006–2025 · barras = IC 95% (bootstrap)")
ax.legend()
plt.show()

In [ ]:
# Mapas por microrregião: anomalia média nas safras de El Niño e de La Niña (descritivo)
fig, axes = plt.subplots(1, 2, figsize=(16, 8), layout="constrained")
for ax, f in zip(axes, ("El Niño", "La Niña")):
    anos = fases.index[fases["fase"] == f]
    media = ANOM_SAFRA.reindex(anos).mean().rename("anom")
    g = micro.merge(media, left_on="CD_MICRO", right_index=True, how="left")
    micro.plot(ax=ax, color="#eeeeee", linewidth=0)
    g.dropna(subset=["anom"]).plot(ax=ax, column="anom", cmap="BrBG", vmin=-25, vmax=25, edgecolor="white", linewidth=0.1)
    ax.set_title(f"Safras de {f} (n = {len(anos)}): anomalia média da chuva da safra")
    ax.axis("off")
fig.colorbar(plt.cm.ScalarMappable(cmap="BrBG", norm=plt.Normalize(-25, 25)), ax=list(axes), shrink=0.6,
             label="anomalia (% da média)")
plt.show()

In [ ]:
# Intensidade: ONI de pico da safra × anomalia regional (cada ponto é uma safra)
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6), layout="constrained")
for ax, reg in zip(axes, ["Cerrado", "Amazônia", "Pampa"]):
    d = pd.DataFrame({"oni": fases["oni_pico"], "anom": reg_pct[reg], "fase": fases["fase"], "rot": fases["rotulo"]}).dropna()
    for f, cor in cores.items():
        s_ = d[d["fase"] == f]
        ax.scatter(s_["oni"], s_["anom"], color=cor, label=f, s=40)
    for _, r in d.iterrows():
        ax.annotate(r["rot"], (r["oni"], r["anom"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
    rho = stats.spearmanr(d["oni"], d["anom"]).statistic
    ax.axhline(0, color="black", lw=0.6); ax.axvline(0, color="black", lw=0.6)
    ax.set_title(f"{reg} · ρ(ONI de pico, anomalia) = {rho:+.2f} (n = {len(d)})")
    ax.set_xlabel("ONI de pico na safra"); ax.set_ylabel("anomalia da chuva da safra (%)")
axes[0].legend(fontsize=8)
plt.show()

In [ ]:
# Cerrado por subperíodo (ligação com a seção 18) e pela fase da safra ANTERIOR
lin = []
for sp, meses_cal, mn in [("safra inteira (set–abr)", P["meses_safra"], 7), ("início (set–nov)", [9, 10, 11], 3),
                          ("dez–abr", [12, 1, 2, 3, 4], 4)]:
    rp = regional_pct(anomalia_pct(meses_cal, mn), P_LAG["divisao_principal"]).reindex(fases.index)["Cerrado"]
    c = comparar_fases(rp)
    fase_ant = fases["fase"].shift(1)                      # fase da safra anterior
    d_ant = pd.DataFrame({"y": rp, "f": fase_ant}).dropna()
    lin.append({"subperíodo": sp,
                "El Niño (%)": c["El Niño"]["media"], "La Niña (%)": c["La Niña"]["media"],
                "El Niño − La Niña": c["dif"]["media"], "p diferença": c["dif"]["p"],
                "após El Niño (%)": d_ant.loc[d_ant["f"] == "El Niño", "y"].mean(),
                "após La Niña (%)": d_ant.loc[d_ant["f"] == "La Niña", "y"].mean()})
pd.DataFrame(lin).round(2)

### Leitura da seção 19
| Bioma | El Niño | La Niña | Diferença (p) | Leitura |
|---|---|---|---|---|
| **Pampa** | +18,9% | **−18,4%** (robusto) | 37 p.p. (**p = 0,001**) | efeito claro e **simétrico**; a **intensidade** acompanha (ρ = +0,74 entre o ONI de pico e a chuva): 2023/24 (El Niño forte) +62%, 2022/23 (La Niña) −45% |
| **Amazônia** | −14,4% (p = 0,03) | +5,7% | −20 p.p. (**p = 0,04**) | **assimétrico**: o efeito do El Niño é cerca de 2,5 vezes o do La Niña |
| Caatinga | −0,6% | +18,0% (p = 0,03) | −19 p.p. (p = 0,14) | La Niña mais chuvoso; poucas microrregiões com soja (11) |
| **Cerrado** | +0,8% | +3,3% | −3 p.p. (p = 0,61) | **sem diferença na média por fase**, coerente com a seção 18 (efeitos de sinais opostos ao longo da safra se cancelam). Os **El Niños fortes** (2015/16, 2023/24) tiveram cerca de 10% menos chuva, e os fracos não: o efeito pode depender da intensidade |
| Mata Atlântica | +6,8% | −2,5% | +9 p.p. (p = 0,17) | sem efeito claro |

**Diferença em relação ao teste por substitutas (seção 17):** lá o Pampa não passou (p ≈ 0,19), e aqui o efeito dele é o mais forte. As duas análises medem coisas diferentes:
- **correlação mês a mês:** todos os meses e todas as intensidades, com muito ruído mensal, só nas safras até 2020;
- **comparação por fases:** o total da safra (que reduz o ruído) e o contraste entre os **extremos** (El Niño × La Niña), com todas as safras de 2006 a 2025.

O efeito do ENSO no Pampa aparece com clareza **nos eventos**, e não como uma relação linear mês a mês.

**Cerrado e a safra anterior** (última tabela, descritiva):
- no **início** da safra, depois de uma safra de El Niño a chuva foi 10,5% menor, e depois de uma de La Niña, 13,9% maior;
- isso é coerente com a relação defasada negativa das seções 12 e 18 (o ENSO de 7 a 12 meses antes);
- são poucas safras por grupo, e não foi feito teste formal.

**Cautelas:**
- poucas safras por fase (El Niño 7, La Niña 9, neutro 4) e eventos de intensidades muito diferentes na mesma fase;
- a classificação de fase é uma escolha (definição da NOAA e o mínimo de 4 dos 8 meses), registrada no D-06;
- o p-valor por fase compara com a média de todas as safras, e não com as neutras (são só 4).

## 20. Conclusões da Parte 3
1. **Teste por substitutas (seção 17):**
   - a relação defasada do **Cerrado** (cerca de 7 meses) é a única, entre os biomas, que passa depois de descontar a busca e a autocorrelação (p ≈ 0,015 nas duas famílias), e o **MT** e o **GO** passam individualmente;
   - a **Amazônia** não passa;
   - a correção também entre regiões leva o Cerrado a 0,07–0,08, então a evidência dele se apoia no conjunto: teste, repetição em 2021–2025 e coerência entre as UFs. Proposta de revisão dos níveis em D-04.
2. **Subperíodos (seção 18):**
   - a hipótese do início das chuvas **não foi apoiada**;
   - no Cerrado, o efeito **muda de sinal** ao longo da safra (positivo e simultâneo em setembro; negativo e defasado de dezembro a abril);
   - a relação defasada **não é** um retrato do estado atual do ENSO (correlação parcial praticamente igual).
3. **Fases (seção 19):**
   - o **Pampa** tem o efeito mais claro em termos de chuva (La Niña −18%, El Niño +19%), proporcional à intensidade do evento;
   - a **Amazônia** responde sobretudo ao El Niño (−14%);
   - no **Cerrado**, a média por fase não mostra diferença, mas os El Niños fortes foram mais secos.

**Mensagem geral:** o ENSO se relaciona com a chuva da safra de formas **diferentes em cada região**:
- **no Sul**, por eventos e de forma simultânea;
- **na Amazônia**, sobretudo pelo El Niño;
- **no Cerrado**, por uma combinação de efeitos de sinais opostos ao longo da safra, com um componente defasado que é o resultado estatisticamente mais robusto.

Isso reforça tratar o ENSO **por região e por fase da safra**, e não como um único efeito nacional.

**Acréscimos das seções 21 e 22:**
- **Sensibilidade (seção 21):** o resultado do Cerrado e as comparações por fase são **robustos** a todas as escolhas de método testadas (Spearman/Pearson, anomalia z/%, completude de 70–90%, com ou sem estações substitutas).
- **Outras variáveis (seção 22):**
  - os **dias secos** confirmam a relação defasada do Cerrado (7 meses);
  - a **temperatura máxima** responde ao El Niño no Norte e no Centro-Oeste mesmo sem a tendência de aquecimento;
  - o sinal dos **dias quentes** vinha da tendência;
  - nenhuma variável por safra (veranico, início das chuvas, dias quentes) se sustenta após a correção FDR.

## 21. Sensibilidade às escolhas de método
**Pergunta:** os resultados principais dependem das escolhas que fizemos? Cada variante muda **uma** escolha em relação à base (Spearman, anomalia z por mês do calendário, completude mensal de 80%, só estações dentro da microrregião):

| Variante | O que muda | Por que testar |
|---|---|---|
| V1 | correlação de **Pearson** | a de Spearman foi escolhida pela assimetria da chuva; Pearson é a referência usual |
| V2 | anomalia mensal em **% da média do mês** | a z padroniza pela variabilidade; a % só pela média |
| V3 | completude mínima de **70%** | mais meses e microrregiões, com mais risco de soma subestimada |
| V4 | completude mínima de **90%** | menos meses, mas mais íntegros |
| V5 | **incluir estações substitutas** | aumenta a cobertura (microrregiões sem estação própria), mas repete a mesma estação em várias microrregiões |

**O que é verificado:**
1. **Resultado-chave (Cerrado):** ρ com 7 meses e janela de 3, melhor combinação e p do teste por substitutas (as duas famílias da seção 17);
2. **ρ de cada bioma** na defasagem escolhida no D-03;
3. **comparações por fase** (Pampa e Amazônia): só nas variantes que alteram o total da safra (V3, V4 e V5). V1 e V2 não mudam o total.

**Critérios (fixados antes da execução):**
- **Cerrado robusto:** em todas as variantes, mesmo sinal, \|ρ(7 m, j3)\| ≥ 0,20, melhor defasagem entre 5 e 9 meses e p (fase) < 0,10;
- **fases robustas:** mesmo sinal, com magnitude ≥ 50% da base.

**Validação:** a variante base (V0) precisa reproduzir os números das seções 12, 17 e 19.

In [ ]:
mapa_completo = mapa_estacoes_cd_micro(PROCESSED / "clima_diario.parquet").dropna(subset=["CD_MICRO"])
mensal_todos = (clima.groupby(["codigo_estacao", "mes_ref"])
                     .agg(chuva=("prec_total", "sum"), dias=("prec_total", "count")).reset_index())


def construir_variante(completude: float = 0.80, so_poligono: bool = True, tipo: str = "z") -> tuple[pd.DataFrame, pd.DataFrame]:
    mp = mapa_completo[mapa_completo["origem"] == "dentro_do_poligono"] if so_poligono else mapa_completo
    ms = mensal_todos[mensal_todos["dias"] >= completude * mensal_todos["mes_ref"].dt.days_in_month]
    cm = (ms.merge(mp[["codigo_estacao", "CD_MICRO"]], on="codigo_estacao")
            .groupby(["CD_MICRO", "mes_ref"])["chuva"].mean().reset_index())
    cm["mes"] = cm["mes_ref"].dt.month
    g_ = cm.groupby(["CD_MICRO", "mes"])["chuva"]
    cm["n_anos"] = g_.transform("count")
    if tipo == "z":
        cm["anom"] = (cm["chuva"] - g_.transform("mean")) / g_.transform("std")
    else:                                     # "pct": % da média do mês
        cm["anom"] = 100 * (cm["chuva"] / g_.transform("mean") - 1)
    cm.loc[cm["n_anos"] < P["min_anos_climatologia"], "anom"] = np.nan
    an = (cm[cm["mes"].isin(P["meses_safra"])].pivot(index="mes_ref", columns="CD_MICRO", values="anom").sort_index())
    an = an[[c for c in an.columns if c in producao.index]].dropna(axis=1, how="all")
    return an, cm


def regionais_de(an: pd.DataFrame, nome: str) -> pd.DataFrame:
    rot = divisoes[nome].reindex(an.columns).dropna()
    return pd.DataFrame({reg: an[mics].mean(axis=1) for reg, mics in rot.groupby(rot).groups.items()})


def rho_todas_m(X: np.ndarray, Y: pd.DataFrame, metodo: str) -> np.ndarray:
    if metodo == "spearman":
        return rho_todas(X, Y)
    out = np.full((Y.shape[1], X.shape[1]), np.nan)
    for j, reg in enumerate(Y.columns):
        y = Y[reg].to_numpy()
        ok = ~np.isnan(y) & ~np.isnan(X).any(axis=1)
        if ok.sum() < 24:
            continue
        yy = (y[ok] - y[ok].mean()) / y[ok].std()
        xx = (X[ok] - X[ok].mean(0)) / X[ok].std(0)
        out[j] = (xx * yy[:, None]).mean(0)
    return out


def avaliar_variante(nome_var: str, completude=0.80, so_poligono=True, tipo="z", metodo="spearman") -> tuple[dict, dict]:
    an, cm = construir_variante(completude, so_poligono, tipo)
    Y = regionais_de(an, P_LAG["divisao_principal"])[(ano_safra.reindex(an.index) <= P_LAG["safra_ultima_exploracao"]).values]
    rho = rho_todas_m(matriz_indice(oni_ref, Y.index), Y, metodo)
    j = list(Y.columns).index("Cerrado")
    k7 = COMBOS.index((7, 3))
    kbest = int(np.nanargmax(np.abs(rho[j])))
    nulos = {fam: np.array([np.nanmax(np.abs(rho_todas_m(matriz_indice(pd.Series(s, index=oni_ref.index), Y.index),
                                                              Y[["Cerrado"]], metodo))) for s in arr])
             for fam, arr in subs.items()}
    mx = np.nanmax(np.abs(rho[j]))
    p_ = {fam: (1 + (nulos[fam] >= mx).sum()) / (len(nulos[fam]) + 1) for fam in nulos}
    lin = {"variante": nome_var, "micros Cerrado": int(divisoes[P_LAG["divisao_principal"]].reindex(an.columns).eq("Cerrado").sum()),
           "ρ Cerrado (7 m, j3)": rho[j, k7], "melhor (L, j)": f"{COMBOS[kbest][0]} m, j{COMBOS[kbest][1]}",
           "máx|ρ| Cerrado": mx, "p fase": p_["fase"], "p deslocamento": p_["deslocamento"]}
    # ρ de cada bioma na defasagem do D-03
    d03 = resultado_bioma.dropna(subset=["defasagem (meses)"]).set_index("região")
    for reg in Y.columns:
        if reg in d03.index and not np.isnan(d03.loc[reg, "defasagem (meses)"]):
            kk = COMBOS.index((int(d03.loc[reg, "defasagem (meses)"]), int(d03.loc[reg, "janela"])))
            lin[f"ρ {reg} ({int(d03.loc[reg, 'defasagem (meses)'])} m)"] = rho[list(Y.columns).index(reg), kk]
    # fases (total da safra em %)
    fas = {}
    if tipo == "z" and metodo == "spearman":
        cm_ = cm.rename(columns={"anom": "_"})
        global chuva_micro
        _orig = chuva_micro
        chuva_micro = cm_
        try:
            rp = regional_pct(anomalia_pct(P["meses_safra"], 7), P_LAG["divisao_principal"],
                              micros_validas=an.columns).reindex(fases.index)
        finally:
            chuva_micro = _orig
        for reg, f in [("Pampa", "La Niña"), ("Pampa", "El Niño"), ("Amazônia", "El Niño")]:
            fas[f"{reg} · {f} (%)"] = comparar_fases(rp[reg], n_perm=2000)[f]["media"]
    return lin, fas


variantes = [("V0 · base", {}), ("V1 · Pearson", {"metodo": "pearson"}), ("V2 · anomalia %", {"tipo": "pct"}),
             ("V3 · completude 70%", {"completude": 0.70}), ("V4 · completude 90%", {"completude": 0.90}),
             ("V5 · com estações substitutas", {"so_poligono": False})]
res_sens, res_fases = [], []
for nome_var, kw in variantes:
    lin, fas = avaliar_variante(nome_var, **kw)
    res_sens.append(lin)
    if fas:
        res_fases.append({"variante": nome_var, **fas})
sens = pd.DataFrame(res_sens).set_index("variante")
sens_fases = pd.DataFrame(res_fases).set_index("variante")

# Avaliação com os critérios fixados
base = sens.loc["V0 · base"]
lag_best = sens["melhor (L, j)"].str.extract(r"^(\d+)").astype(int)[0]
crit = pd.DataFrame({
    "mesmo sinal": np.sign(sens["ρ Cerrado (7 m, j3)"]) == np.sign(base["ρ Cerrado (7 m, j3)"]),
    "|ρ| ≥ 0,20": sens["ρ Cerrado (7 m, j3)"].abs() >= 0.20,
    "melhor defasagem 5–9 m": lag_best.between(5, 9),
    "p fase < 0,10": sens["p fase"] < 0.10,
})
crit["Cerrado robusto"] = crit.all(axis=1)
print("CERRADO ROBUSTO EM TODAS AS VARIANTES" if crit["Cerrado robusto"].all() else
      f"Cerrado NÃO robusto em: {list(crit.index[~crit['Cerrado robusto']])}")
sens.round(3)

In [ ]:
crit

In [ ]:
# Comparações por fase nas variantes que mudam o total da safra
bf = sens_fases.loc["V0 · base"]
robusto_fases = sens_fases.apply(lambda r: (np.sign(r) == np.sign(bf)) & (r.abs() >= 0.5 * bf.abs()), axis=1)
print("Fases robustas em todas as variantes:", robusto_fases.all().to_dict())
sens_fases.round(1)

### Leitura da seção 21
**Validação:** a variante base (V0) reproduz os números das seções 12, 17 e 19 (Cerrado ρ = −0,294, p = 0,016 / 0,014; Pampa com La Niña −18,4%; Amazônia com El Niño −14,4%).

**Resultado-chave (Cerrado):** robusto em **todas** as variantes pelos critérios fixados.

| Variante | ρ (7 m, j3) | Melhor combinação | p fase / deslocamento |
|---|---|---|---|
| V0 · base | −0,294 | 7 m, j3 | 0,016 / 0,014 |
| V1 · Pearson | −0,281 | 7 m, j1 | 0,013 / 0,028 |
| V2 · anomalia % | −0,282 | 8 m, j1 | 0,033 / 0,042 |
| V3 · completude 70% | −0,291 | 7 m, j3 | 0,018 / 0,014 |
| V4 · completude 90% | −0,284 | 7 m, j3 | 0,017 / 0,014 |
| V5 · com substitutas (91 micros) | −0,291 | 8 m, j1 | 0,018 / 0,014 |

A defasagem fica sempre entre **7 e 8 meses**, e o p continua abaixo de 0,05 nas duas famílias em todas as variantes.

**Outros biomas** (ρ na defasagem do D-03): os sinais não mudam em nenhuma variante. As maiores variações são no Pampa (+0,21 a +0,27) e na Mata Atlântica (+0,11 a +0,21, este último com Pearson), que continua fraca.

**Fases (total da safra):** robustas nas três variantes que alteram o total:
- Pampa com La Niña: −17,6% a −18,4%;
- Pampa com El Niño: +18,6% a +19,6%;
- Amazônia com El Niño: −9,4% a −14,4%. O menor valor é o da V5: as microrregiões que entram com estações substitutas atenuam o efeito, mas ele continua acima de 50% da base.

**Correção feita durante a análise:** na primeira execução, a V5 deu resultados idênticos à base nas fases, porque a função de agregação filtrava pelas microrregiões da base. A função foi corrigida para usar as microrregiões de cada variante; os resultados das seções anteriores não mudaram.

**Conclusão:** os resultados principais **não dependem** das escolhas de correlação (Spearman/Pearson), do tipo de anomalia (z/%), do limiar de completude (70–90%) nem da inclusão das estações substitutas.

## 22. Outras variáveis climáticas
**Por que:** o ENSO não muda só o total de chuva. Para a soja, também importam a **distribuição da chuva** (veranicos), o **calor** e a **data de início da estação chuvosa**, que define a janela de plantio.

**Variáveis** (calculadas a partir do clima diário de cada estação, com a mesma regra de completude de 80%):

| Variável | Escala | Definição |
|---|---|---|
| fração de dias secos | mensal | dias com menos de 2 mm (`dia_seco`) ÷ dias com dado |
| Tmax média | mensal | média da temperatura máxima diária |
| fração de dias quentes | mensal e safra | dias com Tmax acima do **percentil 90 da própria estação** (set–abr) ÷ dias com dado |
| maior veranico | safra | maior sequência de dias secos entre 1/dez e 31/mar |
| início das chuvas | safra | primeiro dia após 1/set com ≥ 20 mm em 3 dias **sem** 10 dias secos seguidos nos 30 dias seguintes; só onde há estação chuvosa definida (Cerrado, Amazônia, MT, GO) |

**Escolhas e por quê:**
- **Dia quente relativo à estação (P90),** e não um limite fixo (como 35 °C): a Tmax do pipeline é o máximo das médias horárias e fica **subestimada** (mediana de 29,8 °C). Um limite relativo não é afetado por esse viés.
- **Início das chuvas só nas regiões de monção:** o critério foi validado (mediana em **14/out** no Centro-Oeste, como na literatura). No Sul ele dá 10/set, porque lá não há estação chuvosa definida e o critério só pega a primeira chuva.

**Testes (fixados antes da execução):**
- **(a) Mensais:** varredura de defasagens e **teste por substitutas** (seção 17), com a mesma regra (passa / indício / não passa), nas safras de exploração.
- **(b) Por safra:** comparação por fases (como na seção 19) e correlação de Spearman com o **ONI de pico** da safra e com o **ONI médio de mai–jul antes da safra** (ENSO antecedente), com p por permutação (10.000). **Robusto** se p < 0,05. Usa todas as safras (2006–2025), porque é uma comparação descritiva.

In [ ]:
clima_ext = pd.read_parquet(PROCESSED / "clima_diario.parquet", columns=["codigo_estacao", "data", "prec_total", "temp_max", "dia_seco"])
clima_ext["data"] = pd.to_datetime(clima_ext["data"])
clima_ext["mes_ref"] = clima_ext["data"].dt.to_period("M")
na_safra = clima_ext["data"].dt.month.isin(P["meses_safra"])
p90 = clima_ext[na_safra].groupby("codigo_estacao")["temp_max"].quantile(0.90).rename("p90")
clima_ext = clima_ext.merge(p90, on="codigo_estacao")
clima_ext["dia_quente"] = (clima_ext["temp_max"] > clima_ext["p90"]).astype(float)
clima_ext.loc[clima_ext["temp_max"].isna(), "dia_quente"] = np.nan
mapa_poligono = mapa[["codigo_estacao", "CD_MICRO"]]


def anomalia_mensal_var(coluna: str, sem_tendencia: bool = False) -> pd.DataFrame:
    # anomalia z por mês do calendário de uma variável diária agregada (média) por estação-mês -> microrregião
    m = (clima_ext.dropna(subset=[coluna]).groupby(["codigo_estacao", "mes_ref"])[coluna]
                  .agg(["mean", "count"]).reset_index())
    m = m[m["count"] >= P["completude_mensal"] * m["mes_ref"].dt.days_in_month]
    cm = (m.merge(mapa_poligono, on="codigo_estacao").groupby(["CD_MICRO", "mes_ref"])["mean"].mean().reset_index())
    cm["mes"] = cm["mes_ref"].dt.month
    g_ = cm.groupby(["CD_MICRO", "mes"])["mean"]
    cm["z"] = (cm["mean"] - g_.transform("mean")) / g_.transform("std")
    cm.loc[g_.transform("count") < P["min_anos_climatologia"], "z"] = np.nan
    an = cm[cm["mes"].isin(P["meses_safra"])].pivot(index="mes_ref", columns="CD_MICRO", values="z").sort_index()
    an = an[[c for c in an.columns if c in producao.index]].dropna(axis=1, how="all")
    if sem_tendencia:   # remove a tendência linear de cada microrregião (ex.: aquecimento de longo prazo)
        t = np.arange(len(an), dtype=float)
        for c in an.columns:
            ok = an[c].notna().to_numpy()
            if ok.sum() > 24:
                a, b = np.polyfit(t[ok], an[c].to_numpy()[ok], 1)
                an[c] = an[c] - (a * t + b)
    return an


def teste_generico(an: pd.DataFrame, nome_div: str) -> tuple[pd.DataFrame, np.ndarray, list]:
    Y = regionais_de(an, nome_div)
    Y = Y[(ano_safra.reindex(Y.index) <= P_LAG["safra_ultima_exploracao"]).values]
    rho = rho_todas(matriz_indice(oni_ref, Y.index), Y)
    mx, best = np.nanmax(np.abs(rho), axis=1), np.nanargmax(np.abs(rho), axis=1)
    nulos = {fam: np.array([np.nanmax(np.abs(rho_todas(matriz_indice(pd.Series(s, index=oni_ref.index), Y.index), Y)), axis=1)
                            for s in arr]) for fam, arr in subs.items()}
    lin = []
    for j, reg in enumerate(Y.columns):
        p_ = {fam: (1 + (nulos[fam][:, j] >= mx[j]).sum()) / (len(nulos[fam]) + 1) for fam in nulos}
        n_sig = sum(v < 0.05 for v in p_.values())
        lin.append({"região": reg, "máx|ρ|": mx[j], "ρ": rho[j, best[j]],
                    "defasagem, janela": f"{COMBOS[best[j]][0]} m, j{COMBOS[best[j]][1]}",
                    "ρ em (7 m, j3)": rho[j, COMBOS.index((7, 3))],
                    "p fase": p_["fase"], "p deslocamento": p_["deslocamento"],
                    "resultado": "passa" if n_sig == 2 else ("indício" if n_sig == 1 else "não passa")})
    return pd.DataFrame(lin), rho, list(Y.columns)


VARS_MENSAIS = {"fração de dias secos": "dia_seco", "Tmax média": "temp_max", "fração de dias quentes": "dia_quente"}
an_vars = {nome: anomalia_mensal_var(col) for nome, col in VARS_MENSAIS.items()}
# checagem acrescentada depois da primeira execução: a temperatura tem tendência de aquecimento
an_vars["Tmax média (sem tendência)"] = anomalia_mensal_var("temp_max", sem_tendencia=True)
an_vars["fração de dias quentes (sem tendência)"] = anomalia_mensal_var("dia_quente", sem_tendencia=True)
an_vars_com_chuva = {"chuva (referência, seção 17)": anom, **an_vars}
res_mensal, rho_mensal = [], {}
for nome, an in an_vars_com_chuva.items():
    t, r, regs = teste_generico(an, P_LAG["divisao_principal"])
    res_mensal.append(t.assign(variável=nome))
    rho_mensal[nome] = (r, regs)
tab_mensal = pd.concat(res_mensal)
tab_mensal.pivot(index="região", columns="variável", values=["ρ", "defasagem, janela", "resultado"]).reindex(
    columns=[(m, v) for m in ["ρ", "defasagem, janela", "resultado"] for v in an_vars_com_chuva])

In [ ]:
# Perfil de defasagem no Cerrado (janela 3): as variáveis contam a mesma história que a chuva?
fig, ax = plt.subplots(figsize=(10, 4.2), layout="constrained")
for nome, (r, regs) in rho_mensal.items():
    j = regs.index("Cerrado")
    ks = [k for k, (L, w) in enumerate(COMBOS) if w == 3]
    ax.plot([COMBOS[k][0] for k in ks], r[j, ks], marker="o", label=nome, lw=2 if "chuva" in nome else 1.3)
ax.axhline(0, color="black", lw=0.7)
ax.set_xlabel("defasagem (meses)"); ax.set_ylabel("ρ (ONI defasado, janela 3 × anomalia)")
ax.set_title("Cerrado · perfil de defasagem por variável (exploração, safras até 2020)")
ax.legend(fontsize=8)
plt.show()

In [ ]:
# (b) Variáveis por safra
def maior_veranico(g: pd.DataFrame) -> pd.Series:
    out = {}
    s = g.set_index("data")["dia_seco"]
    for ano in range(s.index.year.min(), s.index.year.max() + 1):
        idx = pd.date_range(pd.Timestamp(ano, 12, 1), pd.Timestamp(ano + 1, 3, 31), freq="D")
        v = s.reindex(idx)
        if v.notna().mean() < P["completude_mensal"]:
            continue
        seco = v.fillna(0).to_numpy()               # dia sem dado interrompe a sequência (conservador)
        maior = atual = 0
        for d in seco:
            atual = atual + 1 if d == 1 else 0
            maior = max(maior, atual)
        out[ano + 1] = maior
    return pd.Series(out, dtype=float)


def inicio_chuvas(g: pd.DataFrame, max_dias: int = 150, acum: float = 20.0, seco: int = 10, janela: int = 30) -> pd.Series:
    out = {}
    s = g.set_index("data")["prec_total"]
    for ano in range(s.index.year.min(), s.index.year.max() + 1):
        idx = pd.date_range(pd.Timestamp(ano, 9, 1), periods=max_dias + janela, freq="D")
        v = s.reindex(idx)
        if v.iloc[:max_dias].notna().mean() < P["completude_mensal"]:
            continue
        p, falta = v.fillna(0).to_numpy(), v.isna().to_numpy()
        a3 = np.array([p[d:d + 3].sum() for d in range(max_dias)])
        for d in range(max_dias):
            if a3[d] >= acum and not falta[d:d + 3].any():
                corrida, ok = 0, True
                for x in p[d + 3:d + 3 + janela]:
                    corrida = corrida + 1 if x < 1.0 else 0
                    if corrida >= seco:
                        ok = False
                        break
                if ok:
                    out[ano + 1] = d
                    break
    return pd.Series(out, dtype=float)


def frac_quentes_safra(g: pd.DataFrame) -> pd.Series:
    g = g[g["data"].dt.month.isin(P["meses_safra"])].dropna(subset=["dia_quente"])
    sf = np.where(g["data"].dt.month >= 9, g["data"].dt.year + 1, g["data"].dt.year)
    agg = g.groupby(sf)["dia_quente"].agg(["mean", "count"])
    return agg.loc[agg["count"] >= P["completude_mensal"] * 242, "mean"]


def por_safra(func) -> pd.DataFrame:
    est = clima_ext.groupby("codigo_estacao")[["data", "prec_total", "dia_seco", "dia_quente"]].apply(func)
    est = est.rename_axis(["codigo_estacao", "safra"]).rename("valor").reset_index() if isinstance(est, pd.Series) else \
          est.stack().rename_axis(["codigo_estacao", "safra"]).rename("valor").reset_index()
    micro_ = est.merge(mapa_poligono, on="codigo_estacao").groupby(["CD_MICRO", "safra"])["valor"].mean().unstack(0)
    return micro_[[c for c in micro_.columns if c in anom.columns]]


VARS_SAFRA = {"maior veranico dez–mar (dias)": maior_veranico, "início das chuvas (dias após 1/set)": inicio_chuvas,
              "fração de dias quentes na safra": frac_quentes_safra}
safra_vars = {nome: por_safra(f) for nome, f in VARS_SAFRA.items()}
for nome, d in safra_vars.items():
    print(f"{nome}: {d.shape[1]} microrregiões, safras {int(d.index.min())}–{int(d.index.max())}")

In [ ]:
oni_antecedente = pd.Series({s_: oni_m[(oni_m.index.year == s_ - 1) & oni_m.index.month.isin([5, 6, 7])].mean()
                             for s_ in fases.index})


def rho_perm(x: pd.Series, y: pd.Series, n: int = 10000, seed: int = 42) -> tuple[float, float, int]:
    d = pd.concat([x, y], axis=1).dropna()
    r = stats.spearmanr(d.iloc[:, 0], d.iloc[:, 1]).statistic
    rng = np.random.default_rng(seed)
    perm = np.array([stats.spearmanr(rng.permutation(d.iloc[:, 0].to_numpy()), d.iloc[:, 1]).statistic for _ in range(n)])
    return r, (1 + (np.abs(perm) >= abs(r)).sum()) / (n + 1), len(d)


REGIOES_INICIO = {"Cerrado", "Amazônia", "MT", "GO"}
linhas = []
for nome, d in safra_vars.items():
    reg_b = regional_pct.__wrapped__(d, P_LAG["divisao_principal"]) if hasattr(regional_pct, "__wrapped__") else None
    rot_b, rot_u = divisoes[P_LAG["divisao_principal"]], divisoes[P_LAG["divisao_sensibilidade"]]
    series_ = {}
    for rotulos, regs in ((rot_b, ["Cerrado", "Amazônia", "Pampa", "Mata Atlântica", "Caatinga"]), (rot_u, ["MT", "GO", "RS", "PR"])):
        rot = rotulos.reindex(d.columns).dropna()
        for reg in regs:
            mics = rot.index[rot == reg]
            if len(mics):
                series_[reg] = d[mics].mean(axis=1).reindex(fases.index)
    for reg, y in series_.items():
        if "início" in nome and reg not in REGIOES_INICIO:
            continue
        if y.notna().sum() < 10:
            continue
        c = comparar_fases(y, n_perm=5000)
        r_pico, p_pico, n_ = rho_perm(fases["oni_pico"], y)
        r_ant, p_ant, _ = rho_perm(oni_antecedente, y)
        linhas.append({"variável": nome, "região": reg, "n safras": n_, "média": y.mean(),
                       "El Niño − média": c["El Niño"]["media"] - y.mean(), "La Niña − média": c["La Niña"]["media"] - y.mean(),
                       "p dif. EN×LN": c["dif"]["p"],
                       "ρ ONI de pico": r_pico, "p (pico)": p_pico, "ρ ONI mai–jul antes": r_ant, "p (antecedente)": p_ant})
tab_safra = pd.DataFrame(linhas)
tab_safra["robusto (p<0,05)"] = tab_safra[["p dif. EN×LN", "p (pico)", "p (antecedente)"]].min(axis=1) < 0.05
# correção acrescentada depois da primeira execução: BH sobre todos os p-valores da tabela (3 por linha)
_cols_p = ["p dif. EN×LN", "p (pico)", "p (antecedente)"]
_todos = tab_safra[_cols_p].to_numpy().ravel()
_bh = multipletests(_todos, method="fdr_bh")[1].reshape(-1, 3)
tab_safra["p mínimo corrigido (BH)"] = _bh.min(axis=1)
tab_safra["robusto após BH"] = tab_safra["p mínimo corrigido (BH)"] < 0.05
tab_safra.round(3)

### Leitura da seção 22
**Checagens acrescentadas depois da primeira execução** (registradas por transparência):
1. **Tendência:** a temperatura sobe ao longo dos anos, e o ONI também teve tendência de alta entre 2004 e 2020 (correlação de +0,20 com o tempo). Duas séries com tendência se correlacionam mesmo sem relação causal, e as séries substitutas não reproduzem esse alinhamento. Por isso o teste foi refeito com as variáveis de temperatura **sem a tendência linear** de cada microrregião.
2. **Correção FDR (Benjamini–Hochberg)** sobre os 69 p-valores da tabela das variáveis por safra (23 linhas × 3 testes).

**(a) Variáveis mensais** (teste por substitutas, safras até 2020):

| Variável | Cerrado | Amazônia | Caatinga | Mata Atlântica | Pampa |
|---|---|---|---|---|---|
| fração de dias secos | **passa** (+0,32; 7 m) | **passa** (+0,41; 3 m) | não | não | não |
| Tmax média, **sem tendência** | **passa** (+0,39) | **passa** (+0,43) | **passa** (+0,42) | não | não |
| fração de dias quentes, **sem tendência** | não | não | indício | não | não |

- **Os dias secos são o espelho da chuva:** o Cerrado responde com a **mesma defasagem de 7 meses** (mais dias secos depois de El Niño). É uma confirmação independente do resultado principal, com outra variável.
- **Temperatura:** o El Niño se relaciona com **mais calor** no Norte e no Centro-Oeste (cerca de 3 a 5 meses de defasagem), e o sinal **resiste à remoção da tendência**. É o sinal mais forte entre as variáveis testadas.
- **Dias quentes:** com a tendência, passavam em quatro biomas; sem ela, quase nada passa. O sinal vinha, em grande parte, do **aquecimento de longo prazo**, e não do ENSO.
- **Pampa:** nenhuma variável mensal passa, coerente com a seção 17 (no Sul, o efeito aparece nos eventos, seção 19).

**(b) Variáveis por safra:** **nenhum resultado se sustenta após a correção FDR** (o menor p corrigido é 0,36). Os p < 0,05 isolados são compatíveis com o acaso, dado o número de testes:
- início das chuvas na Amazônia mais tardio depois de El Niño antecedente (ρ = +0,52; p = 0,025);
- dias quentes no Cerrado (ρ = +0,50; p = 0,03).

Ficam como **indícios** a investigar com séries mais longas. O maior veranico (dez–mar) não mostra relação com o ENSO em nenhuma região.

**Limitação:** a Tmax do pipeline é o máximo das médias horárias e fica subestimada. Isso não afeta as anomalias (que comparam cada estação com ela mesma), mas os valores absolutos de temperatura não devem ser interpretados.